# RA hand X-ray severity grading (Mild / Moderate / Severe)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, f1_score, recall_score,
                             classification_report, accuracy_score,
                             cohen_kappa_score, roc_auc_score, balanced_accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from warnings import filterwarnings
filterwarnings('ignore')
import json, os, re
from pathlib import Path
from PIL import Image
import pandas as pd

In [ ]:
import random

SEED = 42

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
imgSize = 384
root = "/content/rheumatoid_arthritis/dataset"


In [ ]:
!pip install -q kaggle
from google.colab import files
uploaded = files.upload()

In [ ]:
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)

_up = globals().get("uploaded") or {}
if _up:
    _kname = next(iter(_up))
    shutil.copy(_kname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("Kaggle key installed from:", _kname)
elif os.path.exists('/root/.kaggle/kaggle.json'):
    print("Kaggle key already present.")
else:
    raise FileNotFoundError("No kaggle.json uploaded: run the cell above and choose your kaggle.json file.")


In [ ]:
!kaggle datasets download -d sindhuvunna/rheumatoid-arthritis-detection
import zipfile
with zipfile.ZipFile('/content/rheumatoid-arthritis-detection.zip', 'r') as z:
    z.extractall('/content/rheumatoid_arthritis')
print("Dataset extracted.")

---
# Preprocessing section



In [ ]:
import subprocess, sys, glob, shutil, time, math
import cv2
from concurrent.futures import ThreadPoolExecutor
DATA_VARIANT = "clahe"
WORK_DIR  = "/content/ra_preproc"
CLAHE_CLIP   = 2.0
CLAHE_TILE   = 8
LONG_SIDE    = None
JPEG_QUALITY = 95
SKIP_CLASSES = []
DETECTOR_WEIGHTS = None
DETECT_ON  = "clahe"
YOLO_IMGSZ = 640
CONF_MIN   = 0.25
CONF_ROI   = 0.50
ROI_PAD    = 0.10
SAVE_PATCHES = False
PATCH_SIZE, PATCH_CONF, PATCH_PAD = 128, 0.90, 0.25


assert DATA_VARIANT in ("raw", "clahe", "roi"), "DATA_VARIANT must be raw, clahe or roi"
SRC_ROOT = root
assert "ra_preproc" not in SRC_ROOT, "root already points to processed data - re-run the cell that sets root first"
assert os.path.isdir(SRC_ROOT), f"{SRC_ROOT} not found - run the Kaggle download cells above first"

SPLITS     = ["train", "val", "test"]
IMG_EXT    = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
OUT_CLAHE  = f"{WORK_DIR}/clahe/dataset"
OUT_ROI    = f"{WORK_DIR}/clahe_roi/dataset"
OUT_PATCH  = f"{WORK_DIR}/joint_patches"
BOXES_CSV  = f"{WORK_DIR}/joint_boxes.csv"
os.makedirs(WORK_DIR, exist_ok=True)

def list_images(root_dir, splits=SPLITS, skip=SKIP_CLASSES):
    """Return a DataFrame: split, label, rel (path relative to root_dir)."""
    rows_ = []
    for s in splits:
        for cls in sorted(os.listdir(os.path.join(root_dir, s))):
            d = os.path.join(root_dir, s, cls)
            if not os.path.isdir(d) or cls in skip:
                continue
            for f in sorted(os.listdir(d)):
                if Path(f).suffix.lower() in IMG_EXT:
                    rows_.append((s, cls, f"{s}/{cls}/{f}"))
    return pd.DataFrame(rows_, columns=["split", "label", "rel"])

src_df = list_images(SRC_ROOT)
print(f"DATA_VARIANT = {DATA_VARIANT}")
print(src_df.groupby(["split", "label"]).size().unstack(fill_value=0))

##  CLAHE



In [ ]:
def load_gray(path, long_side=LONG_SIDE):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        img = np.array(Image.open(path).convert("L"))
    if img.ndim == 3:
        img = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2GRAY)
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    if long_side and max(img.shape) > long_side:
        s = long_side / max(img.shape)
        img = cv2.resize(img, (round(img.shape[1] * s), round(img.shape[0] * s)), interpolation=cv2.INTER_AREA)
    return img

def apply_clahe(gray, clip=CLAHE_CLIP, tile=CLAHE_TILE):
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(gray)

def preprocess(path):
    return apply_clahe(load_gray(path))

def save_jpg(gray, out_path, quality=JPEG_QUALITY):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), [cv2.IMWRITE_JPEG_QUALITY, quality])

def clahe_name(rel):
    return str(Path(rel).with_suffix(".jpg"))

In [ ]:
if not (DATA_VARIANT != "raw"):
    print("skipped (DATA_VARIANT is raw)")
else:
    sample = src_df.groupby("label").head(1).head(4)
    fig, ax = plt.subplots(2, len(sample), figsize=(4 * len(sample), 8))
    ax = np.atleast_2d(ax).reshape(2, -1)
    for j, (_, r) in enumerate(sample.iterrows()):
        g = load_gray(os.path.join(SRC_ROOT, r.rel))
        ax[0, j].imshow(g, cmap="gray"); ax[0, j].set_title(f"{r.label} - original"); ax[0, j].axis("off")
        ax[1, j].imshow(apply_clahe(g), cmap="gray"); ax[1, j].set_title("CLAHE"); ax[1, j].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
if not (DATA_VARIANT != "raw"):
    print("skipped (DATA_VARIANT is raw)")
else:
    def _do(rel):
        out = os.path.join(OUT_CLAHE, clahe_name(rel))
        if os.path.exists(out):
            return "skipped"
        try:
            save_jpg(preprocess(os.path.join(SRC_ROOT, rel)), out)
            return "ok"
        except Exception as e:
            return f"FAILED {rel}: {e}"

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=4) as ex:
        results = list(tqdm(ex.map(_do, src_df.rel), total=len(src_df), desc="CLAHE"))
    fails = [r for r in results if r.startswith("FAILED")]
    print(f"done in {(time.time()-t0)/60:.1f} min | ok {results.count('ok')} | skipped {results.count('skipped')} | failed {len(fails)}")
    for f in fails[:10]: print(" ", f)

    out_df = list_images(OUT_CLAHE)
    print(out_df.groupby(["split", "label"]).size().unstack(fill_value=0))
    assert len(out_df) == len(src_df) - len(fails), "image count differs from the source"

### CHECK 1: CLAHE


In [ ]:

def _local_contrast(g, blk=32):
    h, w = (g.shape[0] // blk) * blk, (g.shape[1] // blk) * blk
    tiles = g[:h, :w].astype(np.float32).reshape(h // blk, blk, w // blk, blk)
    return float(tiles.std(axis=(1, 3)).mean())

def _entropy(g):
    p = np.bincount(g.ravel(), minlength=256).astype(np.float64)
    p /= p.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

if DATA_VARIANT == "raw":
    print("CHECK 1 skipped: DATA_VARIANT is 'raw' (no CLAHE in this run).")
else:
    smp = src_df.sample(min(60, len(src_df)), random_state=SEED)
    recs, missing = [], []
    for _, r in smp.iterrows():
        raw = load_gray(os.path.join(SRC_ROOT, r.rel))
        outp = os.path.join(OUT_CLAHE, clahe_name(r.rel))
        im = cv2.imread(outp, cv2.IMREAD_UNCHANGED)
        if im is None:
            missing.append(outp)
            continue
        three = (im.ndim == 3 and im.shape[2] == 3 and im.dtype == np.uint8)
        g = im[:, :, 0] if im.ndim == 3 else im
        ref = apply_clahe(raw)
        same = g.shape == ref.shape
        recs.append({
            "three_channel_uint8": three,
            "channel_spread": int(np.abs(im[:, :, 0].astype(int) - im[:, :, 2].astype(int)).max()) if im.ndim == 3 else 0,
            "same_size": same,
            "diff_vs_recomputed": float(np.abs(g.astype(int) - ref.astype(int)).mean()) if same else np.nan,
            "contrast_raw": _local_contrast(raw), "contrast_clahe": _local_contrast(g),
            "entropy_raw": _entropy(raw), "entropy_clahe": _entropy(g),
            "extreme_raw": float(((raw == 0) | (raw == 255)).mean()),
            "extreme_clahe": float(((g == 0) | (g == 255)).mean()),
        })
    d1 = pd.DataFrame(recs)
    assert len(d1) > 0, "no CLAHE files could be read: run the CLAHE build cell first"
    ratio = d1.contrast_clahe / d1.contrast_raw.clip(lower=1e-6)
    checks1 = [
        ("every sampled CLAHE file exists", len(missing) == 0, f"{len(missing)} missing"),
        ("files are 3-channel uint8", bool(d1.three_channel_uint8.all()), ""),
        ("grey stored as RGB (channels identical, JPEG tolerance 4)", int(d1.channel_spread.max()) <= 4, f"max spread {int(d1.channel_spread.max())}"),
        ("file == CLAHE(original): mean |diff| < 3 grey levels (CLAHE ran exactly once, with these settings)",
         bool(d1.same_size.all()) and float(d1.diff_vs_recomputed.max()) < 3.0, f"worst {float(d1.diff_vs_recomputed.max()):.2f}"),
        ("local contrast raised in >= 90% of images", float((d1.contrast_clahe > d1.contrast_raw).mean()) >= 0.90,
         f"{float((d1.contrast_clahe > d1.contrast_raw).mean()):.0%} of images, mean ratio {float(ratio.mean()):.2f}x"),
        ("grey-level entropy not lowered in >= 90% of images", float((d1.entropy_clahe >= d1.entropy_raw - 0.02).mean()) >= 0.90,
         f"mean {d1.entropy_raw.mean():.2f} -> {d1.entropy_clahe.mean():.2f} bits"),
        ("no new clipping: extra pure-black/white pixels < 2 percentage points", float((d1.extreme_clahe - d1.extreme_raw).max()) < 0.02,
         f"worst {float((d1.extreme_clahe - d1.extreme_raw).max()) * 100:+.2f} points (negative = fewer saturated pixels)"),
    ]
    for name, ok_, detail in checks1:
        print(("PASS" if ok_ else "FAIL"), "|", name, "|", detail)
    print("\nCHECK 1 RESULT:", "CLAHE is working." if all(c[1] for c in checks1) else
          "CLAHE has a problem: re-run the CLAHE build cell (delete the OUT_CLAHE folder first if settings changed).")

    r0 = smp.iloc[0]
    g_raw = load_gray(os.path.join(SRC_ROOT, r0.rel))
    g_cl = cv2.imread(os.path.join(OUT_CLAHE, clahe_name(r0.rel)), cv2.IMREAD_GRAYSCALE)
    fig, ax = plt.subplots(2, 2, figsize=(10, 8))
    ax[0, 0].imshow(g_raw, cmap="gray"); ax[0, 0].set_title("original"); ax[0, 0].axis("off")
    ax[0, 1].imshow(g_cl, cmap="gray"); ax[0, 1].set_title("after CLAHE (what the classifier reads)"); ax[0, 1].axis("off")
    ax[1, 0].hist(g_raw.ravel(), bins=64, color="gray"); ax[1, 0].set_title("grey-level histogram: original")
    ax[1, 1].hist(g_cl.ravel(), bins=64, color="gray"); ax[1, 1].set_title("histogram: CLAHE (should be flatter / wider)")
    plt.tight_layout(); plt.show()


##  YOLO joint detector



In [ ]:

if DATA_VARIANT == "roi":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)
    print("ultralytics ready")
else:
    print("skipped (DATA_VARIANT is not roi)")

In [ ]:

ANNOT_IMAGES = "/content/annotated/images"
ANNOT_LABELS = "/content/annotated/labels"
CLASS_NAMES  = ["Joint"]
TRAIN_VARIANTS = ["clahe", "raw"]
YOLO_BASE    = "yolov8n.pt"
EPOCHS, BATCH = 100, 8
VAL_FRACTION  = 0.2

def build_yolo_dataset(variant):
    base = Path(WORK_DIR) / f"yolo_{variant}"
    imgs = sorted(p for p in Path(ANNOT_IMAGES).iterdir() if p.suffix.lower() in IMG_EXT)
    assert imgs, f"no images in {ANNOT_IMAGES}"
    rng = random.Random(SEED); rng.shuffle(imgs)
    n_val = max(1, round(len(imgs) * VAL_FRACTION))
    split = {"val": imgs[:n_val], "train": imgs[n_val:]}
    for s, files_ in split.items():
        (base / "images" / s).mkdir(parents=True, exist_ok=True)
        (base / "labels" / s).mkdir(parents=True, exist_ok=True)
        for p in files_:
            lab = Path(ANNOT_LABELS) / (p.stem + ".txt")
            if not lab.exists():
                print("no label for", p.name, "- skipped"); continue
            out = base / "images" / s / (p.stem + ".jpg")
            if variant == "clahe":
                save_jpg(preprocess(p), str(out))
            else:
                save_jpg(load_gray(p), str(out))
            shutil.copy(lab, base / "labels" / s / (p.stem + ".txt"))
    yaml_path = base / "data.yaml"
    yaml_path.write_text(f"path: {base}\ntrain: images/train\nval: images/val\nnames:\n" +
                         "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)))
    print(f"{variant}: {len(split['train'])} train / {len(split['val'])} val images -> {yaml_path}")
    return str(yaml_path)

In [ ]:
if not (DATA_VARIANT == "roi"):
    print("skipped (DATA_VARIANT is not roi)")
else:

    from ultralytics import YOLO
    import torch

    det_rows, det_weights = [], {}
    if DETECTOR_WEIGHTS is None:
        for v in TRAIN_VARIANTS:
            data_yaml = build_yolo_dataset(v)
            m = YOLO(YOLO_BASE)
            m.train(data=data_yaml, imgsz=YOLO_IMGSZ, epochs=EPOCHS, batch=BATCH, workers=2,
                    patience=30, seed=SEED, deterministic=True, project=f"{WORK_DIR}/runs", name=f"joint_{v}",
                    exist_ok=True, verbose=False)
            best = f"{WORK_DIR}/runs/joint_{v}/weights/best.pt"
            met = YOLO(best).val(data=data_yaml, imgsz=YOLO_IMGSZ, verbose=False)
            ap = met.box.all_ap                      # [classes, 10] = AP at IoU 0.50, 0.55, ... 0.95
            det_rows.append({"variant": v, "precision": float(met.box.mp), "recall": float(met.box.mr),
                             "mAP@0.50": float(ap[:, 0].mean()), "mAP@0.75": float(ap[:, 5].mean()),
                             "mAP@0.50:0.95": float(met.box.map)})
            det_weights[v] = best
            del m; torch.cuda.empty_cache()
        det_table = pd.DataFrame(det_rows).set_index("variant")
        print(det_table.round(4))
        best_variant = det_table["mAP@0.75"].idxmax()
        DETECTOR_WEIGHTS, DETECT_ON = det_weights[best_variant], best_variant
        print(f"\nUsing the '{best_variant}' detector: {DETECTOR_WEIGHTS}   (DETECT_ON = '{DETECT_ON}')")
    else:
        print("Using your weights:", DETECTOR_WEIGHTS, "| DETECT_ON =", DETECT_ON)

##  detect joints in every image



In [ ]:
if not (DATA_VARIANT == "roi"):
    print("skipped (DATA_VARIANT is not roi)")
else:
    from ultralytics import YOLO
    det = YOLO(DETECTOR_WEIGHTS)
    det_root = OUT_CLAHE if DETECT_ON == "clahe" else SRC_ROOT
    det_df = list_images(det_root)
    det_df["key"] = det_df.rel.str.replace(r"\.[^.]+$", "", regex=True)

    done = set()
    if os.path.exists(BOXES_CSV):
        old = pd.read_csv(BOXES_CSV)
        done = set(old["key"].unique())
        print(f"resuming: {len(done)} images already processed")
    todo = det_df[~det_df.key.isin(done)].reset_index(drop=True)
    print(f"{len(todo)} images to detect")

    CHUNK = 128
    header_needed = not os.path.exists(BOXES_CSV)
    for i in tqdm(range(0, len(todo), CHUNK), desc="detecting"):
        part = todo.iloc[i:i + CHUNK]
        paths = [os.path.join(det_root, r) for r in part.rel]
        rows = []
        for (_, meta), r in zip(part.iterrows(), det.predict(paths, imgsz=YOLO_IMGSZ, conf=CONF_MIN,
                                                             max_det=60, stream=True, verbose=False)):
            h, w = r.orig_shape
            b = r.boxes
            if len(b) == 0:
                rows.append(dict(split=meta.split, label=meta.label, key=meta.key, k=-1, x1=np.nan, y1=np.nan,
                                 x2=np.nan, y2=np.nan, conf=np.nan, cls=-1, W=w, H=h))
                continue
            xyxyn, conf, cls = b.xyxyn.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()
            for k in range(len(b)):
                rows.append(dict(split=meta.split, label=meta.label, key=meta.key, k=k,
                                 x1=xyxyn[k, 0], y1=xyxyn[k, 1], x2=xyxyn[k, 2], y2=xyxyn[k, 3],
                                 conf=float(conf[k]), cls=int(cls[k]), W=w, H=h))
        pd.DataFrame(rows).to_csv(BOXES_CSV, mode="a", header=header_needed, index=False)
        header_needed = False

    boxes = pd.read_csv(BOXES_CSV)
    print("boxes saved to", BOXES_CSV, "| rows:", len(boxes))

In [ ]:
if not (DATA_VARIANT == "roi"):
    print("skipped (DATA_VARIANT is not roi)")
else:

    boxes = pd.read_csv(BOXES_CSV)
    valid = boxes[boxes.k >= 0]
    n_conf = valid[valid.conf >= CONF_ROI].groupby("key").size()
    summary = (boxes.drop_duplicates("key")[["key", "split", "label"]]
               .assign(n_joints=lambda d: d.key.map(n_conf).fillna(0).astype(int)))
    print(summary.groupby("label").n_joints.describe()[["count", "mean", "min", "50%", "max"]].round(1))
    print("\nimages with NO joint at conf >= %.2f: %d of %d" % (CONF_ROI, (summary.n_joints == 0).sum(), len(summary)))

##  hand-ROI dataset and optional joint patches



In [ ]:
def union_roi(df_img, w, h, conf_min=CONF_ROI, pad=ROI_PAD):
    d = df_img[(df_img.k >= 0) & (df_img.conf >= conf_min)]
    if d.empty:
        return None
    x1, y1, x2, y2 = d.x1.min() * w, d.y1.min() * h, d.x2.max() * w, d.y2.max() * h
    px, py = (x2 - x1) * pad, (y2 - y1) * pad
    x1, y1, x2, y2 = max(0, x1 - px), max(0, y1 - py), min(w, x2 + px), min(h, y2 + py)
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))

def build_roi_dataset(boxes_df, clahe_root, out_root):
    fallbacks, n = [], 0
    for key, g in tqdm(boxes_df.groupby("key"), desc="ROI crops"):
        src = None
        for ext in (".jpg", ".jpeg", ".png"):
            if os.path.exists(os.path.join(clahe_root, key + ext)):
                src = os.path.join(clahe_root, key + ext); break
        if src is None:
            fallbacks.append((key, "missing CLAHE image")); continue
        img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
        h, w = img.shape
        box = union_roi(g, w, h)
        if box is None:
            fallbacks.append((key, "no confident joint - kept whole image"))
            crop = img
        else:
            crop = img[box[1]:box[3], box[0]:box[2]]
        save_jpg(crop, os.path.join(out_root, key + ".jpg")); n += 1
    pd.DataFrame(fallbacks, columns=["key", "reason"]).to_csv(os.path.join(os.path.dirname(out_root), "roi_fallbacks.csv"), index=False)
    return n, len(fallbacks)

In [ ]:
if not (DATA_VARIANT == "roi"):
    print("skipped (DATA_VARIANT is not roi)")
else:
    assert os.path.isdir(OUT_CLAHE), "run Part 1 first (the ROI crops are cut from the CLAHE images)"
    n, nf = build_roi_dataset(pd.read_csv(BOXES_CSV), OUT_CLAHE, OUT_ROI)
    print(f"ROI images written: {n} | fell back to the whole image: {nf}")
    print(list_images(OUT_ROI).groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:

def square_patch(img, box_n, size=PATCH_SIZE, pad=PATCH_PAD):
    h, w = img.shape
    cx, cy = (box_n[0] + box_n[2]) / 2 * w, (box_n[1] + box_n[3]) / 2 * h
    side = max((box_n[2] - box_n[0]) * w, (box_n[3] - box_n[1]) * h) * (1 + pad)
    side = max(side, 16)
    x1, y1 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x2, y2 = int(round(x1 + side)), int(round(y1 + side))
    pad_l, pad_t, pad_r, pad_b = max(0, -x1), max(0, -y1), max(0, x2 - w), max(0, y2 - h)
    crop = img[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
    if pad_l or pad_t or pad_r or pad_b:
        crop = cv2.copyMakeBorder(crop, pad_t, pad_b, pad_l, pad_r, cv2.BORDER_REPLICATE)
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

def build_patches(boxes_df, clahe_root, out_root, conf_min=PATCH_CONF):
    meta = []
    d = boxes_df[(boxes_df.k >= 0) & (boxes_df.conf >= conf_min)]
    for key, g in tqdm(d.groupby("key"), desc="patches"):
        src = next((os.path.join(clahe_root, key + e) for e in (".jpg", ".jpeg", ".png")
                    if os.path.exists(os.path.join(clahe_root, key + e))), None)
        if src is None: continue
        img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
        for _, r in g.iterrows():
            p = square_patch(img, (r.x1, r.y1, r.x2, r.y2))
            rel = f"{r.split}/{r.label}/{Path(key).name}_j{int(r.k):02d}.png"
            os.makedirs(os.path.dirname(os.path.join(out_root, rel)), exist_ok=True)
            cv2.imwrite(os.path.join(out_root, rel), p)
            meta.append(dict(patch=rel, image=key, split=r.split, inherited_label=r.label, conf=r.conf, joint_k=int(r.k)))
    pd.DataFrame(meta).to_csv(os.path.join(out_root, "patch_metadata.csv"), index=False)
    return len(meta)

In [ ]:
if not (DATA_VARIANT == "roi"):
    print("skipped (DATA_VARIANT is not roi)")
else:
    if SAVE_PATCHES:
        print("patches written:", build_patches(pd.read_csv(BOXES_CSV), OUT_CLAHE, OUT_PATCH))
    else:
        print("SAVE_PATCHES is False - skipped.")

### CHECK 2: YOLO


In [ ]:

print("DATA_VARIANT =", DATA_VARIANT)
if DATA_VARIANT != "roi":
    print("\nYOLO is NOT part of this run: the classifier trains on whole CLAHE images, so there is nothing to verify.")
    print("EXPECTED output of this cell in a 'clahe' run: exactly this message.\n")
    print("To switch YOLO on you need ONE of:")
    print("  (a) a trained joint detector:  set DETECTOR_WEIGHTS = '/path/best.pt'")
    print("  (b) about 100 hand X-rays with one box per joint (YOLO .txt format) in ANNOT_IMAGES / ANNOT_LABELS")
    print(f"Status now: ANNOT_IMAGES exists = {os.path.isdir(ANNOT_IMAGES)} | ANNOT_LABELS exists = {os.path.isdir(ANNOT_LABELS)} | DETECTOR_WEIGHTS = {DETECTOR_WEIGHTS}")
    print("This Kaggle dataset has image-level grades only (no joint boxes). Without your own boxes keep DATA_VARIANT = 'clahe'.")
else:
    from ultralytics import YOLO
    _ok2 = []
    def _c2(name, passed, detail=""):
        print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
        _ok2.append(bool(passed))

    _c2("detector weights file exists", DETECTOR_WEIGHTS is not None and os.path.exists(str(DETECTOR_WEIGHTS)), str(DETECTOR_WEIGHTS))
    _det = YOLO(str(DETECTOR_WEIGHTS))
    _c2("detector classes == CLASS_NAMES", list(_det.names.values()) == list(CLASS_NAMES), str(_det.names))
    if "det_table" in globals():
        print(det_table.round(3).to_string())
        _c2("best mAP@0.50 >= 0.80 (joints are localised reliably)", float(det_table["mAP@0.50"].max()) >= 0.80, f"{float(det_table['mAP@0.50'].max()):.3f}")
        _c2("best mAP@0.75 >= 0.50", float(det_table["mAP@0.75"].max()) >= 0.50, f"{float(det_table['mAP@0.75'].max()):.3f}")
    else:
        print("(det_table not available because you supplied DETECTOR_WEIGHTS: validate them on your own boxed images)")

    boxes_ = pd.read_csv(BOXES_CSV)
    valid_ = boxes_[boxes_.k >= 0]
    n_conf_ = valid_[valid_.conf >= CONF_ROI].groupby("key").size()
    per_img_ = (boxes_.drop_duplicates("key")[["key", "split", "label"]]
                .assign(n=lambda d: d.key.map(n_conf_).fillna(0).astype(int)))
    _c2("images with NO confident joint <= 5%", float((per_img_.n == 0).mean()) <= 0.05, f"{float((per_img_.n == 0).mean()):.1%}")
    _c2("median confident joints per image >= 8", float(per_img_.n.median()) >= 8, f"median {per_img_.n.median():.0f}, max {per_img_.n.max()}")

    fb_path = os.path.join(os.path.dirname(OUT_ROI), "roi_fallbacks.csv")
    fb = pd.read_csv(fb_path) if os.path.exists(fb_path) else pd.DataFrame(columns=["key", "reason"])
    fb_rate = per_img_.assign(fb=per_img_.key.isin(set(fb.key))).groupby("label").fb.mean()
    print("fallback-to-whole-image rate per grade:\n", fb_rate.round(3).to_string())
    _c2("fallback rate similar across grades (spread <= 5 points; a larger spread would leak the label)",
        float(fb_rate.max() - fb_rate.min()) <= 0.05, f"spread {float(fb_rate.max() - fb_rate.min()):.3f}")

    fr = []
    for key_, g_ in boxes_.groupby("key"):
        w_, h_ = float(g_.W.iloc[0]), float(g_.H.iloc[0])
        b_ = union_roi(g_, w_, h_)
        if b_ is not None:
            fr.append((b_[2] - b_[0]) * (b_[3] - b_[1]) / (w_ * h_))
    fr = np.array(fr)
    _c2("ROI covers 25-95% of the image (median): not useless, not cutting off the hand",
        len(fr) > 0 and 0.25 <= float(np.median(fr)) <= 0.95, f"median {float(np.median(fr)) if len(fr) else float('nan'):.2f}")

    pick_ = per_img_.sample(min(3, len(per_img_)), random_state=SEED)
    fig, ax = plt.subplots(1, len(pick_), figsize=(5 * len(pick_), 5), squeeze=False)
    for a_, (_, row_) in zip(ax[0], pick_.iterrows()):
        src_ = next((os.path.join(OUT_CLAHE, row_.key + e) for e in (".jpg", ".jpeg", ".png") if os.path.exists(os.path.join(OUT_CLAHE, row_.key + e))), None)
        if src_ is None:
            a_.axis("off"); continue
        im_ = cv2.cvtColor(cv2.imread(src_), cv2.COLOR_BGR2RGB)
        hh, ww = im_.shape[:2]
        for _, b in boxes_[(boxes_.key == row_.key) & (boxes_.k >= 0) & (boxes_.conf >= CONF_ROI)].iterrows():
            cv2.rectangle(im_, (int(b.x1 * ww), int(b.y1 * hh)), (int(b.x2 * ww), int(b.y2 * hh)), (255, 0, 0), max(2, ww // 300))
        roi_ = union_roi(boxes_[boxes_.key == row_.key], ww, hh)
        if roi_ is not None:
            cv2.rectangle(im_, (roi_[0], roi_[1]), (roi_[2], roi_[3]), (0, 255, 0), max(3, ww // 200))
        a_.imshow(im_); a_.set_title(f"{row_.label} | {row_.n} joints (red) + ROI (green)", fontsize=9); a_.axis("off")
    plt.tight_layout(); plt.show()
    print("\nEXPECTED picture: red boxes sit on finger / wrist joints, the green ROI contains both hands without cutting fingers.")
    print("CHECK 2 RESULT:", "YOLO ROI stage is working." if all(_ok2) else "YOLO stage has a problem: do NOT train on 'roi' data until the FAIL lines are fixed.")


### Appendix: the original Darknet YOLOv7 commands from `Yolov7.ipynb`



In [ ]:
RUN_DARKNET = False
DARKNET_DIR = "/content/darknet"
CFG         = f"{DARKNET_DIR}/cfg_RA/yolov7.cfg"
VARIANT     = "clahe"

if RUN_DARKNET:
    base = Path(WORK_DIR) / f"yolo_{VARIANT}"
    for s, name in [("train", "train.txt"), ("val", "valid.txt")]:
        (base / name).write_text("\n".join(str(p) for p in sorted((base / "images" / s).glob("*.jpg"))))
    (base / "obj.names").write_text("\n".join(CLASS_NAMES))
    (base / "obj.data").write_text(f"classes = {len(CLASS_NAMES)}\ntrain = {base}/train.txt\nvalid = {base}/valid.txt\n"
                                   f"names = {base}/obj.names\nbackup = {base}/backup\n")
    os.makedirs(base / "backup", exist_ok=True)
    os.chdir(DARKNET_DIR)

    os.system(f"./darknet detector calc_anchors {base}/obj.data -num_of_clusters 9 -width 640 -height 640")

    os.system(f"./darknet detector train {base}/obj.data {CFG} -dont_show -map")

    os.system(f"./darknet detector map {base}/obj.data {CFG} {base}/backup/yolov7_best.weights -iou_thresh 0.75")
else:
    print("RUN_DARKNET is False - skipped.")

In [ ]:
if DATA_VARIANT == "clahe":
    root = OUT_CLAHE
elif DATA_VARIANT == "roi":
    root = OUT_ROI
else:
    root = SRC_ROOT
DATA_TAG = {"raw": "", "clahe": "_clahe", "roi": "_roi"}[DATA_VARIANT]

for s in SPLITS:
    assert os.path.isdir(os.path.join(root, s)), f"{root}/{s} is missing - run the preprocessing cells above first"
print("classifier data root:", root)
print("run-name tag:", repr(DATA_TAG))

In [ ]:
trainData = datasets.ImageFolder(root + '/train', transform=transforms.ToTensor())
valData = datasets.ImageFolder(root + '/val', transform=transforms.ToTensor())
testData = datasets.ImageFolder(root + '/test', transform=transforms.ToTensor())

print("Classes:", trainData.classes)
print(f"Train: {len(trainData)}, Val: {len(valData)}, Test: {len(testData)}")
assert trainData.classes == valData.classes == testData.classes, "class mismatch"

## Dataset audit

In [ ]:
rows = []
for path, label in trainData.samples[::5]:
    im = Image.open(path)
    g = np.array(im.convert("L"))
    rows.append({
        "class": trainData.classes[label],
        "w": im.width,
        "h": im.height,
        "mode": im.mode,
        "ext": path.split(".")[-1].lower(),
        "mean": g.mean(),
        "std": g.std(),
        "kb": os.path.getsize(path) / 1024
    })

stats = pd.DataFrame(rows)
for col in ["w", "h", "kb"]:
    print(f"{col}: {dict(stats.groupby('class')[col].median())}")
for col in ["mean", "std"]:
    print(f"{col}: {dict(stats.groupby('class')[col].mean())}")

print("\nFile modes by class:")
print(stats.groupby("class")["mode"].value_counts())
print("\nFile extensions by class:")
print(stats.groupby("class")["ext"].value_counts())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

feats = ["w", "h", "mean", "std", "kb"]
rf = RandomForestClassifier(100, random_state=0)
normal_vs_ra = cross_val_score(rf, stats[feats], (stats["class"] == "Normal").astype(int), cv=5)
print(f"Normal vs RA (metadata only): {normal_vs_ra.mean():.3f}  -> metadata shortcut!")
ra = stats[stats["class"] != "Normal"]
grade_score = cross_val_score(rf, ra[feats], ra["class"], cv=5)
majority_baseline = ra["class"].value_counts(normalize=True).max()
print(f"Grade (metadata only):        {grade_score.mean():.3f}  (majority: {majority_baseline:.3f})")
print("Conclusion: Normal is solved by format; grading needs the image content.")

In [ ]:
def make_grading(ds):
    keep = [c for c in ds.classes if c != "Normal"]
    remap = {ds.class_to_idx[c]: i for i, c in enumerate(keep)}
    ds.samples = [(p, remap[y]) for p, y in ds.samples if y in remap]
    ds.targets = [y for _, y in ds.samples]
    ds.classes = keep
    ds.class_to_idx = {c: i for i, c in enumerate(keep)}
    return ds

trainData, valData, testData = map(make_grading, (trainData, valData, testData))
print(f"Classes: {trainData.classes}")
print(f"Sizes: train {len(trainData)}, val {len(valData)}, test {len(testData)}")
print(f"Train distribution: {np.bincount(trainData.targets)}")

## Duplicate / leakage check (perceptual hash)


In [ ]:
DUP_THRESHOLD = 10
DROP_NEAR_DUPLICATES_FROM_EVAL = False

def dhash_bits(path, hash_size=16):
    im = Image.open(path).convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    a = np.asarray(im, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten().astype(np.float32)

def min_hamming(a, b, chunk=512):

    bs = b.sum(1)
    out = np.empty(len(a), dtype=np.int32)
    for i in range(0, len(a), chunk):
        x = a[i:i + chunk]
        d = x.sum(1)[:, None] + bs[None, :] - 2 * (x @ b.T)
        out[i:i + chunk] = np.rint(d.min(1)).astype(np.int32)
    return out

hashes = {}
for name, ds in [("train", trainData), ("val", valData), ("test", testData)]:
    hashes[name] = np.stack([dhash_bits(p) for p, _ in tqdm(ds.samples, desc=f"hashing {name}", leave=False)])

dup_flags = {}
for a_name, b_name in [("val", "train"), ("test", "train"), ("test", "val")]:
    d = min_hamming(hashes[a_name], hashes[b_name])
    n_dup = int((d <= DUP_THRESHOLD).sum())
    print(f"{a_name:>4} vs {b_name:<5}: {n_dup:4d}/{len(d)} images with a near-duplicate "
          f"({100 * n_dup / len(d):.1f}%), exact matches: {int((d == 0).sum())}, "
          f"median nearest distance: {np.median(d):.0f}")
    dup_flags[(a_name, b_name)] = d <= DUP_THRESHOLD

if DROP_NEAR_DUPLICATES_FROM_EVAL:
    def drop_samples(ds, drop_mask):
        ds.samples = [s for s, drop in zip(ds.samples, drop_mask) if not drop]
        ds.targets = [y for _, y in ds.samples]
    drop_samples(valData, dup_flags[("val", "train")])
    drop_samples(testData, dup_flags[("test", "train")] | dup_flags[("test", "val")])
    print(f"After dropping -> val {len(valData)}, test {len(testData)}")
else:
    print("Nothing removed (DROP_NEAR_DUPLICATES_FROM_EVAL = False). If the % above is not ~0, "
          "your val/test numbers are optimistic; consider a patient-level / de-duplicated split.")

In [ ]:
def compute_mean_std(samples, size):
    s, s2 = torch.zeros(3), torch.zeros(3)
    for path, _ in tqdm(samples, desc="computing mean/std", leave=False):
        im = Image.open(path)
        im.draft("RGB", (size * 2, size * 2))
        x = transforms.functional.to_tensor(im.convert("RGB").resize((size, size)))
        s += x.mean((1, 2))
        s2 += (x ** 2).mean((1, 2))
    mean = s / len(samples)
    std = (s2 / len(samples) - mean ** 2).sqrt()
    return mean.tolist(), std.tolist()

mean, std = compute_mean_std(trainData.samples, imgSize)
print(f"Computed mean: {[round(v, 4) for v in mean]}")
print(f"Computed std:  {[round(v, 4) for v in std]}")

In [ ]:
class RandomAffineBG:

    def __init__(self, degrees, translate, scale, shear=None):
        self.degrees = degrees
        self.translate = translate
        self.scale = scale
        self.shear = shear

    def __call__(self, img):
        arr = np.asarray(img)
        border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
        fill = tuple(int(v) for v in np.median(border, axis=0))
        angle, shift, scl, shr = transforms.RandomAffine.get_params(
            [-self.degrees, self.degrees], self.translate, self.scale,
            list(self.shear) if self.shear else None, list(img.size)
        )
        return transforms.functional.affine(
            img, angle, shift, scl, shr,
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=fill
        )


USE_HFLIP = True

train_aug = [transforms.Resize((imgSize, imgSize), antialias=True)]
if USE_HFLIP:
    train_aug.append(transforms.RandomHorizontalFlip(p=0.5))
train_aug += [
    RandomAffineBG(degrees=12, translate=(0.06, 0.06), scale=(0.88, 1.12), shear=(-4, 4)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
]
transformer = transforms.Compose(train_aug)

valTransformer = transforms.Compose([
    transforms.Resize((imgSize, imgSize), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

trainData.transform = transformer
valData.transform = valTransformer
testData.transform = valTransformer

print("Transforms updated with correct mean/std. Train augmentation:")
print(transformer)


In [ ]:
train_labels = np.array(trainData.targets)
class_counts = np.bincount(train_labels, minlength=len(trainData.classes))

class_weights = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(len(trainData.classes)), y=train_labels),
    dtype=torch.float32
)

sampling_weights = 1.0 / class_counts
train_sample_weights = torch.DoubleTensor([sampling_weights[y] for y in train_labels])
train_sampler = WeightedRandomSampler(train_sample_weights, len(train_sample_weights), replacement=True)

print(f"Class counts: {dict(zip(trainData.classes, class_counts))}")
print(f"Class weights: {dict(zip(trainData.classes, [round(w.item(), 3) for w in class_weights]))}")

In [ ]:
batchSize = 16

trainLoader = DataLoader(trainData, batch_size=batchSize, sampler=train_sampler,
                         num_workers=2, pin_memory=True)
valLoader = DataLoader(valData, batch_size=batchSize, shuffle=False,
                       num_workers=2, pin_memory=True)
testLoader = DataLoader(testData, batch_size=batchSize, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Loaders created: {len(trainLoader)} / {len(valLoader)} / {len(testLoader)} batches")

In [ ]:
MODEL_NAME = "efficientnet_b0"
LOSS_TYPE = "coral"
USE_LOSS_CLASS_WEIGHTS = False
HEAD_DROPOUT = 0.4

assert LOSS_TYPE in ("ce", "coral")

assert trainData.classes == ["Mild", "Moderate", "Severe"], f"class order is not ordinal: {trainData.classes}"

loss_tag = "coral" if LOSS_TYPE == "coral" else ("ce_w" if USE_LOSS_CLASS_WEIGHTS else "ce")
model_name = f"{MODEL_NAME}_{loss_tag}_grading3_seed{SEED}"

class CoralLayer(nn.Module):

    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_features, 1, bias=False)
        self.bias = nn.Parameter(torch.arange(num_classes - 1, 0, -1).float() / (num_classes - 1))
    def forward(self, x):
        return self.fc(x) + self.bias

def make_head(in_features, num_classes):
    return CoralLayer(in_features, num_classes) if LOSS_TYPE == "coral" else nn.Linear(in_features, num_classes)

def logits_to_probs(outputs):

    outputs = outputs.float()
    if LOSS_TYPE == "coral":
        gt = torch.cummin(torch.sigmoid(outputs), dim=1).values
        one, zero = torch.ones_like(gt[:, :1]), torch.zeros_like(gt[:, :1])
        cum = torch.cat([one, gt, zero], dim=1)
        return cum[:, :-1] - cum[:, 1:]
    return torch.softmax(outputs, dim=1)

def decode_probs(probs):

    if LOSS_TYPE == "coral":
        cum = probs.cumsum(1) if torch.is_tensor(probs) else np.cumsum(probs, axis=1)
        return (cum[:, :-1] < 0.5).sum(1)
    return probs.argmax(1)

def build_model(name, num_classes, dropout=HEAD_DROPOUT):

    m = getattr(models, name)(weights="DEFAULT")
    if name.startswith("resnet"):
        m.fc = nn.Sequential(nn.Dropout(dropout), make_head(m.fc.in_features, num_classes))
        head = m.fc
    elif name.startswith("densenet"):
        m.classifier = nn.Sequential(nn.Dropout(dropout), make_head(m.classifier.in_features, num_classes))
        head = m.classifier
    elif name.startswith("efficientnet"):
        m.classifier[0].p = dropout                     # torchvision default is 0.2
        m.classifier[1] = make_head(m.classifier[1].in_features, num_classes)
        head = m.classifier[1]
    elif name.startswith("convnext"):
        m.classifier[2] = make_head(m.classifier[2].in_features, num_classes)
        head = m.classifier[2]
    elif name.startswith("swin"):
        m.head = make_head(m.head.in_features, num_classes)
        head = m.head
    else:
        raise ValueError(f"Unknown model: {name}")

    head_ids = {id(p) for p in head.parameters()}
    backbone = [p for p in m.parameters() if id(p) not in head_ids]
    return m, backbone, list(head.parameters())

seed_all(SEED)
model, backbone_params, head_params = build_model(MODEL_NAME, len(trainData.classes))
model = model.to(device)

HEAD_DIM = len(trainData.classes) - 1 if LOSS_TYPE == "coral" else len(trainData.classes)
model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 3, imgSize, imgSize).to(device))
    print(f"Model output shape: {out.shape}  (expected [2, {HEAD_DIM}])")
    assert out.shape == torch.Size([2, HEAD_DIM]), "output shape mismatch"
    assert torch.allclose(logits_to_probs(out).sum(1), torch.ones(2, device=out.device), atol=1e-4)
    assert decode_probs(logits_to_probs(out)).shape == (2,)
model.train()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"{model_name}: {round(n_params, 2)}M parameters")


## CBAM

In [ ]:

USE_CBAM       = True
CBAM_PLACEMENT = "stages"
CBAM_REDUCTION = 16
CBAM_KERNEL    = 7

assert CBAM_PLACEMENT in ("stages", "last")

class ChannelAttention(nn.Module):

    def __init__(self, ch, reduction=16):
        super().__init__()
        hidden = max(ch // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, ch, 1, bias=False),
        )
        nn.init.zeros_(self.mlp[2].weight)
    def forward(self, x):
        avg = self.mlp(F.adaptive_avg_pool2d(x, 1))
        mx  = self.mlp(F.adaptive_max_pool2d(x, 1))
        return 2.0 * torch.sigmoid(avg + mx)

class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        nn.init.zeros_(self.conv.weight)
    def forward(self, x):
        pooled = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return 2.0 * torch.sigmoid(self.conv(pooled))

class CBAM(nn.Module):
    def __init__(self, ch, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(ch, reduction)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.ca(x)
        return x * self.sa(x)

def add_cbam(m, name, placement=CBAM_PLACEMENT):

    new = []
    if name.startswith("efficientnet"):

        m.eval()
        chans, x = {}, torch.zeros(1, 3, imgSize, imgSize, device=device)
        with torch.no_grad():
            for i, layer in enumerate(m.features):
                x = layer(x)
                chans[i] = x.shape[1]
        last = len(m.features) - 1
        idxs = list(range(1, last)) if placement == "stages" else [last]
        for i in idxs:
            c = CBAM(chans[i], CBAM_REDUCTION, CBAM_KERNEL)
            m.features[i] = nn.Sequential(m.features[i], c)
            new.append(c)
    elif name.startswith("resnet"):
        names = ["layer1", "layer2", "layer3", "layer4"]
        for ln in (names if placement == "stages" else names[-1:]):
            layer = getattr(m, ln)
            blk = layer[-1]
            ch = blk.bn3.num_features if hasattr(blk, "bn3") else blk.bn2.num_features
            c = CBAM(ch, CBAM_REDUCTION, CBAM_KERNEL)
            setattr(m, ln, nn.Sequential(layer, c))
            new.append(c)
    else:
        raise NotImplementedError(f"CBAM insertion is implemented for efficientnet_* and resnet*, not {name}")
    return new

if USE_CBAM:
    n_before = sum(p.numel() for p in model.parameters())
    cbam_modules = add_cbam(model, MODEL_NAME)
    model = model.to(device)


    head_params = list(head_params) + [p for c in cbam_modules for p in c.parameters()]
    assert len(backbone_params) + len(head_params) == len(list(model.parameters())), "parameter groups do not cover the model"

    model_name = model_name + "_cbam" + ("" if CBAM_PLACEMENT == "stages" else "_last")

    model.eval()
    with torch.no_grad():
        out = model(torch.randn(2, 3, imgSize, imgSize).to(device))
    assert out.shape == torch.Size([2, HEAD_DIM]), f"output shape mismatch: {out.shape}"
    model.train()

    added = sum(p.numel() for p in model.parameters()) - n_before
    print(f"CBAM added at {len(cbam_modules)} location(s) ({CBAM_PLACEMENT}) | +{added/1e6:.3f}M parameters")
    print(f"Model output shape: {out.shape} | run name so far: {model_name}")
else:
    print("USE_CBAM is False - model unchanged.")


In [ ]:

REG_TAG = "_v3"
model_name = model_name + DATA_TAG + REG_TAG
print("run name:", model_name)


In [ ]:
class CoralLoss(nn.Module):

    def __init__(self, num_classes):
        super().__init__()
        self.register_buffer("thresholds", torch.arange(num_classes - 1))
    def forward(self, logits, labels):
        levels = (labels.unsqueeze(1) > self.thresholds.unsqueeze(0)).float()
        return F.binary_cross_entropy_with_logits(logits.float(), levels, reduction="none").sum(1).mean()

if LOSS_TYPE == "coral":
    criterion = CoralLoss(len(trainData.classes)).to(device)
    loss_desc = "CORAL (ordinal); class balance comes from the sampler only"
else:

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if USE_LOSS_CLASS_WEIGHTS else None, label_smoothing=0.1)
    loss_desc = f"CrossEntropyLoss + label_smoothing 0.1 (class weights: {USE_LOSS_CLASS_WEIGHTS})"


LR_BACKBONE = {
    "resnet18": 1e-4, "resnet50": 1e-4, "densenet169": 1e-4,
    "efficientnet_b0": 2e-4, "convnext_tiny": 5e-5, "swin_t": 5e-5, "efficientnet_b0_attn": 1e-4
}


HEAD_LR = 5e-4
WEIGHT_DECAY = 5e-2

def split_decay(params):
    return [p for p in params if p.ndim > 1], [p for p in params if p.ndim <= 1]

bb_decay, bb_nodecay = split_decay(backbone_params)
hd_decay, hd_nodecay = split_decay(head_params)

optimizer = optim.AdamW(
    [
        {"params": bb_decay,   "lr": LR_BACKBONE[MODEL_NAME], "weight_decay": WEIGHT_DECAY},
        {"params": bb_nodecay, "lr": LR_BACKBONE[MODEL_NAME], "weight_decay": 0.0},
        {"params": hd_decay,   "lr": HEAD_LR,                 "weight_decay": WEIGHT_DECAY},
        {"params": hd_nodecay, "lr": HEAD_LR,                 "weight_decay": 0.0},
    ]
)

import math
EPOCHS_TOTAL, WARMUP_EPOCHS, MIN_LR_FRAC = 30, 2, 0.02

def lr_lambda(ep):
    if ep < WARMUP_EPOCHS:
        return (ep + 1) / WARMUP_EPOCHS
    t = min(1.0, (ep - WARMUP_EPOCHS) / max(1, EPOCHS_TOTAL - WARMUP_EPOCHS))
    return max(MIN_LR_FRAC, 0.5 * (1.0 + math.cos(math.pi * t)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Loss: {loss_desc}")
print(f"Optimizer: AdamW, backbone LR {LR_BACKBONE[MODEL_NAME]:.0e}, head LR {HEAD_LR:.0e}, weight decay {WEIGHT_DECAY} (none on bias/norm/CORAL thresholds)")

In [ ]:
import os, shutil
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.ismount("/content/drive"):
        print("Drive is already mounted.")
    else:
        if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
            shutil.move("/content/drive", "/content/drive_old")
            print("Moved leftover files to /content/drive_old")
        drive.mount("/content/drive")
    OUT_BASE = Path("/content/drive/MyDrive")
    print(os.listdir("/content/drive/MyDrive")[:10])
else:
    OUT_BASE = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("./outputs")
    print("Not running in Colab: checkpoints and results go to", OUT_BASE.resolve())
OUT_BASE.mkdir(parents=True, exist_ok=True)


In [ ]:
CHECKPOINT_DIR = OUT_BASE / "ra_xray_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / f"{model_name}_best_checkpoint.pth"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / f"{model_name}_last_checkpoint.pth"

start_epoch = 0
bestF1 = -1.0
bestLoss = float("inf")
currentPatience = 0
trainLosses, valLosses, valAccs = [], [], []

print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Best:  {CHECKPOINT_PATH}")
print(f"Last:  {LAST_CHECKPOINT_PATH}")


*Inserted (optional):* keep the YOLO detector weights and joint boxes on Drive so you do not have to train or detect again in a new session.

In [ ]:
if DATA_VARIANT == "roi":
    dst = OUT_BASE / "ra_xray_processed"; dst.mkdir(parents=True, exist_ok=True)
    for f in [BOXES_CSV, os.path.join(WORK_DIR, "clahe_roi", "roi_fallbacks.csv"), DETECTOR_WEIGHTS]:
        if f and os.path.exists(f):
            shutil.copy(f, dst); print("copied", f)
else:
    print("skipped (DATA_VARIANT is not roi)")

In [ ]:
patience = 12
minDelta = 0
currentPatience = 0
bestF1 = -1.0
bestLoss = float('inf')

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
print(f"Early stopping patience: {patience}")
print(f"GradScaler enabled: {scaler.is_enabled()}")

In [ ]:
if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(LAST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        print("GradScaler restored.")

    start_epoch = checkpoint["epoch"]
    bestF1 = checkpoint.get("best_f1", -1.0)
    bestLoss = checkpoint["best_loss"]
    currentPatience = checkpoint["current_patience"]
    trainLosses = checkpoint["train_losses"]
    valLosses = checkpoint["val_losses"]
    valAccs = checkpoint["val_accuracies"]

    print(f"Resumed {checkpoint['model_name']} from epoch {start_epoch + 1}")
    print(f"  best_f1: {bestF1:.4f}, best_loss: {bestLoss:.5f}, patience: {currentPatience}/{patience}")
else:
    print("No previous checkpoint found. Starting fresh.")

### CHECK 3: pre-training checks

In [ ]:

PRECHECK_STRICT = True
_res3 = []
def _c3(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
    _res3.append(bool(passed))

_expected_root = {"raw": SRC_ROOT, "clahe": OUT_CLAHE, "roi": OUT_ROI}[DATA_VARIANT]
_c3("classifier reads the folder that matches DATA_VARIANT", root == _expected_root, root)


_cnt = np.zeros(len(trainData.classes))
_first = None
for _bi, (_xb, _yb) in enumerate(trainLoader):
    if _first is None:
        _first = (_xb, _yb)
    _cnt += np.bincount(_yb.numpy(), minlength=len(trainData.classes))
    if _bi >= 7:
        break
xb, yb = _first
_c3("batch shape [B,3,imgSize,imgSize]", tuple(xb.shape[1:]) == (3, imgSize, imgSize), str(tuple(xb.shape)))
_c3("batch values finite", bool(torch.isfinite(xb).all()), "")
_c3("normalised input: |mean| < 0.8 and 0.5 < std < 1.6", abs(xb.mean().item()) < 0.8 and 0.5 < xb.std().item() < 1.6,
    f"mean {xb.mean().item():.2f}, std {xb.std().item():.2f}")
_share = _cnt / _cnt.sum()
_c3("sampler balances the grades (each share 0.20 - 0.47 over 8 batches)", bool(((_share > 0.20) & (_share < 0.47)).all()),
    str({c: round(float(s), 2) for c, s in zip(trainData.classes, _share)}))


if LOSS_TYPE == "coral":
    _g = torch.tensor([[0.60, 0.35]])
    _p = torch.cat([1 - _g[:, :1], _g[:, :1] - _g[:, 1:], _g[:, 1:]], dim=1)
    _c3("CORAL probabilities are rebuilt correctly from logits", bool(torch.allclose(logits_to_probs(torch.logit(_g)), _p, atol=1e-5)), "")
    _c3("median decoding returns Moderate where argmax would return Mild",
        int(decode_probs(_p).item()) == 1 and int(_p.argmax(1).item()) == 0, "this was the reason Moderate recall stayed ~0.5")


if USE_CBAM:
    if start_epoch == 0:
        _worst = 0.0
        for _c in cbam_modules:
            _x = torch.randn(2, _c.ca.mlp[0].in_channels, 12, 12, device=device)
            with torch.no_grad():
                _worst = max(_worst, (_c(_x) - _x).abs().max().item())
        _c3("CBAM is the identity at initialisation (pretrained features untouched)", _worst < 1e-5,
            f"max |CBAM(x) - x| = {_worst:.1e} over {len(cbam_modules)} modules")
    model.eval()
    model.zero_grad(set_to_none=True)
    _out = model(xb.to(device).float())
    criterion(_out, yb.to(device)).backward()
    _g2 = [c.ca.mlp[2].weight.grad for c in cbam_modules] + [c.sa.conv.weight.grad for c in cbam_modules]
    _c3("CBAM receives gradients (channel-MLP output layer and spatial conv, every stage)",
        all(g is not None and bool(torch.isfinite(g).all()) and g.abs().sum().item() > 0 for g in _g2),
        "expected at the very start: these two layers learn first; the first MLP layer gets gradient only after they move")
    model.zero_grad(set_to_none=True)
    model.train()


_n_opt = sum(p.numel() for g in optimizer.param_groups for p in g["params"])
_n_mod = sum(p.numel() for p in model.parameters())
_c3("optimizer covers all model parameters", _n_opt == _n_mod, f"{_n_opt} vs {_n_mod}")
print("learning rates per group:", [f"{g['lr']:.1e}" for g in optimizer.param_groups],
      "| weight decay per group:", [g["weight_decay"] for g in optimizer.param_groups])

if all(_res3):
    print("\nCHECK 3 RESULT: all pre-training checks passed.")
elif PRECHECK_STRICT:
    raise AssertionError("CHECK 3 failed: fix the FAIL lines above before training.")
else:
    print("\nCHECK 3 RESULT: some checks FAILED (PRECHECK_STRICT is False, continuing).")


## Training loop with overfitting monitoring

In [ ]:
epochs = EPOCHS_TOTAL
print(f"Starting training: {start_epoch} -> {epochs}")

for epoch in range(start_epoch, epochs):
    model.train()
    train_loss = 0.0
    train_correct = train_total = 0

    progress_bar = tqdm(enumerate(trainLoader), total=len(trainLoader),
                        desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch_idx, (inputs, labels) in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        predicted = decode_probs(logits_to_probs(outputs))
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

        progress_bar.set_postfix(loss=train_loss / (batch_idx + 1),
                                 acc=100.0 * train_correct / train_total)

    train_loss /= len(trainLoader)
    train_acc = 100.0 * train_correct / train_total
    trainLosses.append(train_loss)
    model.eval()
    val_loss = 0.0
    val_correct = val_total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in valLoader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            predicted = decode_probs(logits_to_probs(outputs))
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
            all_preds += predicted.cpu().tolist()
            all_labels += labels.cpu().tolist()

    val_loss /= len(valLoader)
    val_acc = 100.0 * val_correct / val_total
    valLoss = val_loss
    valAccs.append(val_acc)
    valLosses.append(val_loss)


    val_f1 = float(f1_score(all_labels, all_preds, average='macro'))
    per_class_recall = recall_score(all_labels, all_preds, average=None,
                                     labels=list(range(len(trainData.classes))), zero_division=0)


    gap = val_loss - train_loss
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:2d}/{epochs} | "
          f"train_loss {train_loss:.4f} val_loss {val_loss:.4f} gap {gap:+.4f} | "
          f"train_acc {train_acc:5.1f}% val_acc {val_acc:5.1f}% | "
          f"macro_f1 {val_f1:.4f} (best {bestF1:.4f}) | "
          f"recall {dict(zip(trainData.classes, [round(float(r), 3) for r in per_class_recall]))} | "
          f"lr {current_lr:.1e}")

    scheduler.step()


    if val_f1 > bestF1 + minDelta:
        bestF1 = val_f1
        bestLoss = val_loss
        currentPatience = 0

        best_checkpoint = {
            "epoch": epoch + 1,
            "model_name": model_name,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_f1": bestF1,
            "best_loss": bestLoss,
            "current_patience": currentPatience,
            "train_losses": trainLosses,
            "val_losses": valLosses,
            "val_accuracies": valAccs,
            "class_names": trainData.classes
        }
        torch.save(best_checkpoint, CHECKPOINT_PATH)
        print(f"  -> NEW BEST: saved checkpoint")
    else:
        currentPatience += 1
        print(f"  -> F1 did not improve ({currentPatience}/{patience})")
    last_checkpoint = {
        "epoch": epoch + 1,
        "model_name": model_name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_f1": bestF1,
        "best_loss": bestLoss,
        "current_patience": currentPatience,
        "train_losses": trainLosses,
        "val_losses": valLosses,
        "val_accuracies": valAccs,
        "class_names": trainData.classes
    }
    torch.save(last_checkpoint, LAST_CHECKPOINT_PATH)
    if currentPatience >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

print("Training complete.")

## Overfitting analysis: train vs validation losses

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(trainLosses, label='train', marker='o', markersize=3)
ax1.plot(valLosses, label='val', marker='s', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

gaps = np.array(valLosses) - np.array(trainLosses)
ax2.plot(gaps, label='val - train', marker='^', markersize=4, color='red')
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss gap')
ax2.set_title('Overfitting gap (positive = overfitting)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/overfitting.png', dpi=100, bbox_inches='tight')
plt.show()

mean_gap = np.mean(gaps[-5:]) if len(gaps) >= 5 else np.mean(gaps)
print(f"Mean gap (last 5): {mean_gap:+.4f}  (0 = perfect, positive = overfitting)")

## Results helper: metrics and export

In [ ]:
from pathlib import Path
import json, copy

RESULTS_DIR = OUT_BASE / "ra_xray_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_NOTES = (f"{model_name}, 3-class grading (no Normal), loss={loss_desc}, "
             f"weighted sampler, moderate augmentation (flip/affine/jitter), identity-init CBAM, weight decay {WEIGHT_DECAY}, head dropout {HEAD_DROPOUT}, median decoding of CORAL, warm-up + cosine LR, imgSize {imgSize}, "
             f"checkpoint on macro-F1, seed {SEED}")
EVAL_TEST = False

class_names = trainData.classes
clinical_rank = {"Mild": 0, "Moderate": 1, "Severe": 2}
rank_of_index = np.array([clinical_rank[c] for c in class_names])

def predict_loader(loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs.to(device))
            all_probs.append(logits_to_probs(outputs).cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)

def expected_calibration_error(y_true, probs, n_bins=10):

    pred = decode_probs(probs)
    conf = probs[np.arange(len(pred)), pred]
    correct = (pred == y_true).astype(float)
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y_true, probs):
    y_pred = decode_probs(probs)
    report = classification_report(y_true, y_pred, target_names=class_names,
                                   output_dict=True, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "qwk": float(cohen_kappa_score(rank_of_index[y_true], rank_of_index[y_pred],
                                       weights="quadratic")),
        "macro_auc": float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro")),
        "within_one_grade": float((np.abs(y_true - y_pred) <= 1).mean()),
        "mae_grade": float(np.abs(rank_of_index[y_true] - rank_of_index[y_pred]).mean()),
        "ece": expected_calibration_error(y_true, probs),
        "per_class": {c: {"precision": report[c]["precision"],
                          "recall": report[c]["recall"],
                          "f1": report[c]["f1-score"],
                          "support": int(report[c]["support"])}
                      for c in class_names},
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

def to_python(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj.item() if hasattr(obj, "item") else str(obj)

def bootstrap_ci(y_true, probs, n_boot=1000, seed=SEED):

    rng = np.random.default_rng(seed)
    y_pred = decode_probs(probs)
    n, K = len(y_true), len(class_names)
    draws = {"macro_f1": [], "qwk": [], **{f"recall_{c}": [] for c in class_names}}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        draws["macro_f1"].append(f1_score(yt, yp, average="macro", zero_division=0))
        draws["qwk"].append(cohen_kappa_score(rank_of_index[yt], rank_of_index[yp], weights="quadratic"))
        rec = recall_score(yt, yp, average=None, labels=list(range(K)), zero_division=0)
        for c, r in zip(class_names, rec):
            draws[f"recall_{c}"].append(r)
    return {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in draws.items()}

def print_ci(ci, metrics):
    print("95% bootstrap CIs:")
    for k, (lo, hi) in ci.items():
        point = metrics[k] if k in ("macro_f1", "qwk") else metrics["per_class"][k[len("recall_"):]]["recall"]
        print(f"  {k:<16} {point:.3f}  [{lo:.3f}, {hi:.3f}]")

print("Results helpers defined.")

In [ ]:
best_checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint["model_state_dict"])
model = model.to(device)

print(f"Loaded best checkpoint from epoch {best_checkpoint['epoch']}")
print(f"  best_f1: {best_checkpoint.get('best_f1', '?'):.4f}")
print(f"  best_loss: {best_checkpoint['best_loss']:.5f}")
y_val, probs_val = predict_loader(valLoader)
metrics_val = compute_metrics(y_val, probs_val)

print("\n=== Validation Results ===")
print(classification_report(y_val, decode_probs(probs_val), target_names=class_names, digits=4))
print(f"Confusion matrix (rows=true, cols=pred):")
print(np.array(metrics_val["confusion_matrix"]))
print(f"Balanced accuracy: {metrics_val['balanced_accuracy']:.4f}")
print(f"Macro-F1:         {metrics_val['macro_f1']:.4f}")
print(f"QWK:              {metrics_val['qwk']:.4f}")
print(f"Macro AUC:        {metrics_val['macro_auc']:.4f}")
print(f"Within one grade: {metrics_val['within_one_grade']:.4f}")
print(f"MAE (grades):     {metrics_val['mae_grade']:.4f}")

ci_val = bootstrap_ci(y_val, probs_val)
print_ci(ci_val, metrics_val)
trainEvalData = copy.copy(trainData)
trainEvalData.transform = valTransformer
trainEvalLoader = DataLoader(trainEvalData, batch_size=batchSize, shuffle=False,
                             num_workers=2, pin_memory=True)
y_tr, probs_tr = predict_loader(trainEvalLoader)
metrics_train = compute_metrics(y_tr, probs_tr)
f1_gap = metrics_train["macro_f1"] - metrics_val["macro_f1"]
print("\n=== Overfitting gap (best checkpoint, no augmentation) ===")
print(f"Macro-F1  train {metrics_train['macro_f1']:.4f} | val {metrics_val['macro_f1']:.4f} | gap {f1_gap:+.4f}")
print(f"QWK       train {metrics_train['qwk']:.4f} | val {metrics_val['qwk']:.4f} | gap {metrics_train['qwk'] - metrics_val['qwk']:+.4f}")
print(f"Accuracy  train {metrics_train['accuracy']:.4f} | val {metrics_val['accuracy']:.4f}")
print(f"ECE (calibration error) train {metrics_train['ece']:.4f} | val {metrics_val['ece']:.4f}")


## Fit diagnosis


In [ ]:

FIT_UNDER_TRAIN_F1 = 0.75
FIT_OVER_GAP       = 0.12
FIT_MILD_GAP       = 0.06

tr_f1, va_f1 = metrics_train["macro_f1"], metrics_val["macro_f1"]
gap_f1 = tr_f1 - va_f1


w = min(8, len(trainLosses))
if w >= 4:
    t = np.arange(w)
    tr_slope = float(np.polyfit(t, np.array(trainLosses[-w:]), 1)[0])
    va_slope = float(np.polyfit(t, np.array(valLosses[-w:]), 1)[0])
else:
    tr_slope = va_slope = 0.0
diverging = (tr_slope < -0.002) and (va_slope > 0.002)

print(f"Macro-F1  train(clean) {tr_f1:.3f} | val {va_f1:.3f} | gap {gap_f1:+.3f}")
print(f"Loss slope over last {w} epochs: train {tr_slope:+.4f}/epoch, val {va_slope:+.4f}/epoch")
if tr_f1 < FIT_UNDER_TRAIN_F1:
    verdict = "UNDER-FIT"
    advice = ("Train longer (raise `EPOCHS_TOTAL`), raise HEAD_LR / LR_BACKBONE a little, lower WEIGHT_DECAY, "
              "or weaken augmentation. Do not add regularisation.")
elif gap_f1 > FIT_OVER_GAP or diverging:
    verdict = "OVER-FIT"
    advice = ("Raise WEIGHT_DECAY (0.05 -> 0.08), raise HEAD_DROPOUT (0.4 -> 0.5), shorten EPOCHS_TOTAL (30 -> 20), strengthen augmentation slightly, "
              "or use the checkpoint from an earlier epoch (the best-F1 checkpoint is already used here).")
elif gap_f1 > FIT_MILD_GAP:
    verdict = "MILD OVER-FIT (acceptable)"
    advice = "Fine for reporting. If you want it smaller: one more seed run, or slightly stronger augmentation."
else:
    verdict = "GOOD FIT"
    advice = "Train and validation agree. Nothing to change."
print(f"\nVERDICT: {verdict}\n{advice}")

### CHECK 4: fit gate (over / under-fit) and CBAM activity


In [ ]:

_res4 = []
def _c4(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
    _res4.append(bool(passed))

tr_f1_, va_f1_ = metrics_train["macro_f1"], metrics_val["macro_f1"]
best_ep_ = int(best_checkpoint["epoch"])
min_val_ep_ = int(np.argmin(valLosses)) + 1
print(f"best-F1 epoch {best_ep_} | lowest val-loss epoch {min_val_ep_} | epochs run {len(valLosses)}")
_c4("not under-fit: clean train macro-F1 >= 0.80", tr_f1_ >= 0.80, f"{tr_f1_:.3f}")
_c4("not over-fit: train - val macro-F1 <= 0.10", tr_f1_ - va_f1_ <= 0.10, f"gap {tr_f1_ - va_f1_:+.3f}")
_c4("val loss did not diverge: last-epoch val loss <= lowest + 0.15", valLosses[-1] <= min(valLosses) + 0.15,
    f"last {valLosses[-1]:.3f} vs lowest {min(valLosses):.3f}")
_c4("best checkpoint is near the lowest val-loss epoch (|diff| <= 8)", abs(best_ep_ - min_val_ep_) <= 8, f"{best_ep_} vs {min_val_ep_}")
_rec = {c: metrics_val["per_class"][c]["recall"] for c in class_names}
_c4("every grade has recall >= 0.55 (Moderate is the weak one)", min(_rec.values()) >= 0.55, str({k: round(v, 3) for k, v in _rec.items()}))
_c4("QWK >= 0.85 and within-one-grade >= 0.98", metrics_val["qwk"] >= 0.85 and metrics_val["within_one_grade"] >= 0.98,
    f"QWK {metrics_val['qwk']:.3f}, within-one {metrics_val['within_one_grade']:.3f}")
_c4("calibration: val ECE <= 0.10 (an over-fit model is over-confident)", metrics_val["ece"] <= 0.10, f"{metrics_val['ece']:.3f}")
print("\nCHECK 4a RESULT:", "GOOD FIT on every criterion." if all(_res4) else
      "At least one criterion failed. See cell above for what to change (WEIGHT_DECAY, HEAD_DROPOUT, EPOCHS_TOTAL, augmentation).")
if USE_CBAM:
    _gates = {i: {"ca": [], "sa": []} for i in range(len(cbam_modules))}
    _hooks = []
    for _i, _c in enumerate(cbam_modules):
        _hooks.append(_c.ca.register_forward_hook(lambda m, inp, out, i=_i: _gates[i]["ca"].append(out.detach().float().cpu())))
        _hooks.append(_c.sa.register_forward_hook(lambda m, inp, out, i=_i: _gates[i]["sa"].append(out.detach().float().cpu())))
    model.eval()
    with torch.no_grad():
        _nv = min(64, len(valData))
        for _s in range(0, _nv, 16):
            model(torch.stack([valData[j][0] for j in range(_s, min(_s + 16, _nv))]).to(device))
    for _h in _hooks:
        _h.remove()
    _dev_all, _bad = [], False
    print("\nCBAM gate statistics on validation images (1.0 = identity, valid range 0..2):")
    for _i in range(len(cbam_modules)):
        _ca, _sa = torch.cat(_gates[_i]["ca"]), torch.cat(_gates[_i]["sa"])
        _bad |= (not bool(torch.isfinite(_ca).all() and torch.isfinite(_sa).all())) or float(_ca.min()) < 0 or float(_ca.max()) > 2 or float(_sa.min()) < 0 or float(_sa.max()) > 2
        _dca, _dsa = float((_ca - 1).abs().mean()), float((_sa - 1).abs().mean())
        _dev_all.append(max(_dca, _dsa))
        print(f"  stage {_i + 1}: channel gate mean|g-1| {_dca:.3f} (min {float(_ca.min()):.2f}, max {float(_ca.max()):.2f}) | spatial gate mean|g-1| {_dsa:.3f}")
    print(("PASS" if not _bad else "FAIL"), "| gates finite and inside (0, 2)")
    _active = max(_dev_all) > 0.02
    print(("PASS" if _active else "WARN"), f"| CBAM has learned something (largest mean|g-1| = {max(_dev_all):.3f}; needs > 0.02)")
    print("EXPECTED: values clearly above 0 in at least some stages. 'Active' only proves CBAM is being used, NOT that it helps:")
    print("  to know that, train the same recipe once with USE_CBAM = False (same seed) and compare val macro-F1 / Moderate recall in all_runs_summary_grading3.csv.")
    del _gates


In [ ]:

results_json = {
    "model_name": model_name,
    "best_epoch": best_checkpoint["epoch"],
    "seed": SEED,
    "notes": RUN_NOTES,
    "loss": loss_tag,
    "validation": metrics_val,
    "train_clean": {k: metrics_train[k] for k in ("accuracy", "macro_f1", "qwk", "ece")},
    "macro_f1_gap_train_minus_val": f1_gap,
    "validation_bootstrap_ci": ci_val,
    "train_loss_history": trainLosses,
    "val_loss_history": valLosses,
    "val_acc_history": valAccs,
}

results_file = RESULTS_DIR / f"{model_name}_results.json"
with open(results_file, 'w') as f:
    json.dump(results_json, f, indent=2, default=to_python)
print(f"Saved: {results_file}")


probs_file = RESULTS_DIR / f"{model_name}_val_probs.npz"
np.savez_compressed(probs_file, y_true=y_val, probs=probs_val)
print(f"Saved: {probs_file}")


summary_row = {
    "model_name": model_name,
    "best_epoch": best_checkpoint["epoch"],
    "seed": SEED,
    "loss": loss_tag,
    "val_accuracy": metrics_val["accuracy"],
    "val_balanced_accuracy": metrics_val["balanced_accuracy"],
    "val_macro_f1": metrics_val["macro_f1"],
    "val_qwk": metrics_val["qwk"],
    "val_macro_auc": metrics_val["macro_auc"],
    "val_within_one_grade": metrics_val["within_one_grade"],
    "val_mae_grade": metrics_val["mae_grade"],
    "val_ece": metrics_val["ece"],
    "train_macro_f1_clean": metrics_train["macro_f1"],
    "macro_f1_gap_train_minus_val": f1_gap,
}
for c in class_names:
    summary_row[f"val_recall_{c}"] = metrics_val["per_class"][c]["recall"]

summary_csv = RESULTS_DIR / "all_runs_summary_grading3.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    df = df[df["model_name"] != model_name]
    df = pd.concat([df, pd.DataFrame([summary_row])], ignore_index=True)
else:
    df = pd.DataFrame([summary_row])
df = df.sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
df.to_csv(summary_csv, index=False)
print(f"Saved: {summary_csv}")
print(f"\nAll runs (sorted by macro-F1):")
print(df[["model_name", "best_epoch", "seed", "val_macro_f1", "val_qwk", "val_recall_Moderate"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cm = np.array(metrics_val["confusion_matrix"])
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(len(class_names)))
axes[0].set_yticks(range(len(class_names)))
axes[0].set_xticklabels(class_names)
axes[0].set_yticklabels(class_names)
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')
axes[0].set_title('Confusion Matrix')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0])

axes[1].plot(trainLosses, label='train', marker='o', markersize=4)
axes[1].plot(valLosses, label='val', marker='s', markersize=4)
axes[1].axvline(best_checkpoint["epoch"] - 1, color='red', linestyle='--', alpha=0.5, label='best')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training & Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)


axes[2].plot(valAccs, label='val accuracy', marker='s', markersize=4, color='orange')
axes[2].axvline(best_checkpoint["epoch"] - 1, color='red', linestyle='--', alpha=0.5, label='best')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Validation Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
curves_file = RESULTS_DIR / f"{model_name}_curves.png"
plt.savefig(curves_file, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {curves_file}")

## Conformal deferral (clinician safety net)


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

TARGET_DEFER = 0.10
K = len(class_names)

def conformal_calibrate(y_cal, probs_cal, alpha):

    scores = 1.0 - probs_cal[np.arange(len(y_cal)), y_cal]
    qhat = np.ones(probs_cal.shape[1])
    for k in range(probs_cal.shape[1]):
        s = scores[y_cal == k]
        if len(s) == 0:
            continue
        level = min(1.0, np.ceil((len(s) + 1) * (1 - alpha)) / len(s))
        qhat[k] = np.quantile(s, level, method="higher")
    return qhat

def conformal_decide(probs, qhat):
    sets = (1.0 - probs) <= qhat[None, :]
    answered = sets.sum(1) == 1
    return sets, answered, sets.argmax(1)

def pick_alpha(y_cal, probs_cal, target=TARGET_DEFER):
    grid = np.round(np.arange(0.01, 0.41, 0.01), 2)
    rates = np.array([1 - conformal_decide(probs_cal, conformal_calibrate(y_cal, probs_cal, a))[1].mean()
                      for a in grid])
    return float(grid[int(np.argmin(np.abs(rates - target)))])

def deferral_report(y, probs, qhat):
    sets, answered, pred = conformal_decide(probs, qhat)
    rep_ = {"answered_frac": float(answered.mean()),
            "deferred_frac": float(1 - answered.mean()),
            "set_coverage": float(sets[np.arange(len(y)), y].mean()),
            "acc_answered": float((pred[answered] == y[answered]).mean()) if answered.any() else float("nan"),
            "per_class": {}}
    for k, c in enumerate(class_names):
        m_all, m_ans = (y == k), (y == k) & answered
        rep_["per_class"][c] = {
            "recall_answered": float((pred[m_ans] == k).mean()) if m_ans.any() else float("nan"),
            "frac_answered": float(m_ans.sum() / max(m_all.sum(), 1)),
        }
    return rep_


rows = []
sss = StratifiedShuffleSplit(n_splits=50, test_size=0.5, random_state=SEED)
for cal_idx, ev_idx in sss.split(probs_val, y_val):
    a = pick_alpha(y_val[cal_idx], probs_val[cal_idx])
    r = deferral_report(y_val[ev_idx], probs_val[ev_idx],
                        conformal_calibrate(y_val[cal_idx], probs_val[cal_idx], a))
    row = {"alpha": a, "deferred": r["deferred_frac"], "acc_answered": r["acc_answered"]}
    for c in class_names:
        row[f"recall_{c}"] = r["per_class"][c]["recall_answered"]
        row[f"answered_{c}"] = r["per_class"][c]["frac_answered"]
    rows.append(row)
conf_cv = pd.DataFrame(rows)

base_rec = metrics_val["per_class"]
print(f"Baseline (answer everything) recall: "
      f"{ {c: round(base_rec[c]['recall'], 3) for c in class_names} }")
print(f"\nConformal deferral, mean +/- std over {len(conf_cv)} val splits (target defer {TARGET_DEFER:.0%}):")
for col in conf_cv.columns:
    print(f"  {col:<20} {conf_cv[col].mean():.3f} +/- {conf_cv[col].std():.3f}")


alpha_star = pick_alpha(y_val, probs_val)
qhat_val = conformal_calibrate(y_val, probs_val, alpha_star)
print(f"\nFinal alpha (calibrated on full val): {alpha_star}, per-class qhat: {np.round(qhat_val, 3)}")

conf_file = RESULTS_DIR / f"{model_name}_conformal.json"
with open(conf_file, "w") as f:
    json.dump({"model_name": model_name, "target_defer": TARGET_DEFER, "alpha": alpha_star,
               "qhat": qhat_val.tolist(), "val_split_summary_mean": conf_cv.mean().to_dict(),
               "val_split_summary_std": conf_cv.std().to_dict()}, f, indent=2, default=to_python)
print(f"Saved: {conf_file}")


# Explainable AI + Vision-Language Model (inserted)


In [ ]:

import cv2
import copy as _copy

XAI_SPLIT   = "val"
XAI_LAYER   = "last"
XAI_TARGET  = "class"
N_PER_GROUP = 2
N_FAITH     = 40

assert XAI_SPLIT in ("val", "test") and XAI_LAYER in ("last", "fine") and XAI_TARGET in ("class", "severity")
assert CHECKPOINT_PATH.exists(), "No best checkpoint found: train first (or run the cell that loads it)."


_ck = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(_ck["model_state_dict"])
model = model.to(device).eval()
del _ck
for _p in model.parameters():
    _p.requires_grad_(True)

if XAI_SPLIT == "val":
    xai_data, xai_y, xai_probs = valData, y_val, probs_val
else:
    xai_data = testData
    xai_y, xai_probs = predict_loader(testLoader)
assert len(xai_data) == len(xai_y) == len(xai_probs), "dataset / prediction length mismatch"
xai_pred = decode_probs(xai_probs)

if "qhat_val" in globals():
    _, xai_answered, _ = conformal_decide(xai_probs, qhat_val)
else:
    xai_answered = np.ones(len(xai_y), dtype=bool)

XAI_MEAN = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
XAI_STD  = torch.tensor(std,  dtype=torch.float32).view(3, 1, 1)

def get_target_layer(m, name, fine=False):
    if name.startswith("efficientnet"):
        return m.features[5] if fine else m.features[-1]
    if name.startswith("resnet"):
        return m.layer3 if fine else m.layer4
    if name.startswith("convnext"):
        return m.features[5] if fine else m.features[-1]
    if name.startswith("densenet"):
        return m.features
    raise NotImplementedError(f"Grad-CAM++ target layer is not defined for {name} (swin_t needs a token-to-grid reshape).")

def class_evidence(out, cls_t):
    out = out.float()
    if LOSS_TYPE == "coral":
        K1 = out.shape[1]
        z_prev = out.gather(1, (cls_t - 1).clamp(min=0)[:, None]).squeeze(1)
        z_curr = out.gather(1, cls_t.clamp(max=K1 - 1)[:, None]).squeeze(1)
        return torch.where(cls_t == 0, -z_curr, torch.where(cls_t == K1, z_prev, -(z_prev + z_curr).abs()))
    return out.gather(1, cls_t[:, None]).squeeze(1)

def grad_campp(x, target=XAI_TARGET, cls=None):

    layer = get_target_layer(model, MODEL_NAME, XAI_LAYER == "fine")
    store = {}
    handle = layer.register_forward_hook(lambda mod, inp, out: store.__setitem__("act", out))
    try:
        model.eval()
        with torch.enable_grad():
            out = model(x.to(device).float())
            probs = logits_to_probs(out)
            cls_t = decode_probs(probs) if cls is None else torch.as_tensor(cls, device=device).long()
            if target == "severity":
                if LOSS_TYPE == "coral":
                    score = out[:, 0].float().sum()
                else:
                    score = (probs * torch.arange(probs.shape[1], device=device)).sum()
            else:
                score = class_evidence(out, cls_t).sum()
            act = store["act"]
            grad = torch.autograd.grad(score, act)[0]
    finally:
        handle.remove()
    A = act.detach().float()
    g = grad.float()
    g2, g3 = g ** 2, g ** 3
    denom = 2.0 * g2 + A.sum(dim=(2, 3), keepdim=True) * g3
    denom = torch.where(denom != 0, denom, torch.ones_like(denom))
    alpha = g2 / denom
    weights = (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * A).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(imgSize, imgSize), mode="bilinear", align_corners=False).squeeze(1)
    mn, mx = cam.amin(dim=(1, 2), keepdim=True), cam.amax(dim=(1, 2), keepdim=True)
    cam = (cam - mn) / (mx - mn + 1e-8)
    return cam.cpu().numpy(), probs.detach().cpu().numpy(), cls_t.cpu().numpy()

def cams_for(indices, bs=8):
    cams, probs, clss = [], [], []
    for s in range(0, len(indices), bs):
        xb = torch.stack([xai_data[int(i)][0] for i in indices[s:s + bs]])
        c, p, k = grad_campp(xb)
        cams.append(c); probs.append(p); clss.append(k)
    return np.concatenate(cams), np.concatenate(probs), np.concatenate(clss)

def denorm_img(x):
    return (x.cpu() * XAI_STD + XAI_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

def overlay_cam(img01, cam, alpha=0.45):
    gray = (img01 * 255).astype(np.uint8)
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return (gray * (1 - alpha) + heat * alpha).astype(np.uint8)

def attention_summary(cam):

    H, W = cam.shape
    if float(cam.max()) <= 0:
        return {"where": "none", "spread": "none", "frac_high": 0.0, "edge_mass": 0.0,
                "sentence": "no usable attention signal (the heat-map is empty)", "short": "empty map"}
    r, c = np.unravel_index(int(cam.argmax()), cam.shape)
    vert  = ["upper", "middle", "lower"][min(int(3 * r / H), 2)]
    horiz = ["left", "centre", "right"][min(int(3 * c / W), 2)]
    where = "centre" if (vert == "middle" and horiz == "centre") else f"{vert}-{horiz}" if vert != "middle" else f"middle-{horiz}"
    frac_hi = float((cam >= 0.5).mean())
    spread = "tightly focused" if frac_hi < 0.12 else ("moderately spread" if frac_hi < 0.30 else "diffuse")
    total = float(cam.sum()) + 1e-8
    b = max(int(0.10 * min(H, W)), 1)
    inner = float(cam[b:H - b, b:W - b].sum())
    edge_mass = 1.0 - inner / total
    sentence = (f"the strongest attention is in the {where} of the displayed image; attention is {spread} "
                f"({frac_hi:.0%} of the image above half intensity); {edge_mass:.0%} of the attention lies in the outer 10% border")
    return {"where": where, "spread": spread, "frac_high": frac_hi, "edge_mass": float(edge_mass),
            "sentence": sentence, "short": f"peak {where}, {spread}"}

def pick_cases(y, pred, n=N_PER_GROUP, seed=SEED):

    rng = np.random.default_rng(seed)
    groups = []
    for k in range(len(class_names)):
        for name_, mask in (("correct", (y == k) & (pred == k)), ("wrong", (y == k) & (pred != k))):
            idx = np.where(mask)[0]
            if len(idx):
                groups.append([(int(i), name_) for i in rng.permutation(idx)[:n]])
    out = []
    for r in range(n):
        for g_ in groups:
            if r < len(g_):
                out.append(g_[r])
    return out

xai_cases = pick_cases(xai_y, xai_pred)
case_idx = [i for i, _ in xai_cases]
cams_c, probs_c, cls_c = cams_for(case_idx)

xai_records = []
for (i, grp), cam, pr, k in zip(xai_cases, cams_c, probs_c, cls_c):
    img = denorm_img(xai_data[i][0])
    summ = attention_summary(cam)
    xai_records.append({"idx": int(i), "group": grp, "true": class_names[int(xai_y[i])], "pred": class_names[int(k)],
                        "probs": {c: float(p) for c, p in zip(class_names, pr)}, "conf": float(pr[int(k)]),
                        "answered": bool(xai_answered[i]), "cam": cam, "img": img,
                        "img_u8": (img * 255).astype(np.uint8), "overlay": overlay_cam(img, cam), "summary": summ,
                        "path": str(xai_data.samples[i][0])})

def show_cases(records, per_fig=4):
    for s in range(0, len(records), per_fig):
        chunk = records[s:s + per_fig]
        fig, axes = plt.subplots(len(chunk), 2, figsize=(8, 3.8 * len(chunk)), squeeze=False)
        for row, rec in zip(axes, chunk):
            row[0].imshow(rec["img"])
            row[0].set_title(f"true {rec['true']} | pred {rec['pred']} ({rec['conf']:.0%})", fontsize=9)
            row[1].imshow(rec["overlay"])
            row[1].set_title(f"Grad-CAM++ | {'answered' if rec['answered'] else 'DEFERRED'} | {rec['summary']['short']}", fontsize=8)
            for a in row:
                a.axis("off")
        plt.tight_layout()
        f = RESULTS_DIR / f"{model_name}_gradcampp_{XAI_SPLIT}_{s // per_fig + 1}.png"
        plt.savefig(f, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {f}")

show_cases(xai_records)

In [ ]:

DEL_STEPS = 10
HWN = imgSize * imgSize

@torch.no_grad()
def prob_curve(x_img, order, cls_k, steps=DEL_STEPS):
    model.eval()
    batch = []
    for s in range(steps + 1):
        k = int(round(HWN * s / steps))
        m = np.zeros(HWN, dtype=bool)
        m[order[:k]] = True
        xi = x_img.clone()
        xi[:, torch.from_numpy(m.reshape(imgSize, imgSize))] = 0.0
        batch.append(xi)
    p = logits_to_probs(model(torch.stack(batch).to(device).float()))[:, cls_k]
    return p.cpu().numpy()

def curve_area(p):
    return float((p[:-1] + p[1:]).sum() / (2 * (len(p) - 1)))

rng_f = np.random.default_rng(SEED)
faith_idx = rng_f.choice(len(xai_y), size=min(N_FAITH, len(xai_y)), replace=False)
faith_rows = []
for s in range(0, len(faith_idx), 8):
    idx_chunk = faith_idx[s:s + 8]
    cams_f, probs_f, cls_f = cams_for(idx_chunk)
    for j, i in enumerate(idx_chunk):
        k = int(cls_f[j])
        x_img = xai_data[int(i)][0]
        p_cam = prob_curve(x_img, np.argsort(-cams_f[j].ravel(), kind="stable"), k)
        p_rnd = prob_curve(x_img, rng_f.permutation(HWN), k)
        faith_rows.append({"idx": int(i), "pred": class_names[k], "auc_cam": curve_area(p_cam), "auc_rand": curve_area(p_rnd),
                           "p_after_20pct_cam": float(p_cam[2]), "p_after_20pct_rand": float(p_rnd[2]),
                           "edge_mass": attention_summary(cams_f[j])["edge_mass"]})
faith = pd.DataFrame(faith_rows)
win = float((faith["auc_cam"] < faith["auc_rand"]).mean())
print(f"Deletion test on {len(faith)} {XAI_SPLIT} images (lower area = the explanation points at what the model really uses)")
print(f"  area under prob-vs-deleted curve : Grad-CAM++ {faith['auc_cam'].mean():.3f} | random {faith['auc_rand'].mean():.3f}")
print(f"  prob after deleting top 20% pixels: Grad-CAM++ {faith['p_after_20pct_cam'].mean():.3f} | random {faith['p_after_20pct_rand'].mean():.3f}")
print(f"  Grad-CAM++ beats random on {win:.0%} of images")
try:
    from scipy.stats import wilcoxon
    pval = float(wilcoxon(faith["auc_cam"], faith["auc_rand"], alternative="less").pvalue)
    print(f"  Wilcoxon signed-rank (cam < random): p = {pval:.4f}")
except Exception as e:
    pval = None
    print("  (Wilcoxon test skipped:", type(e).__name__, ")")
faithful = bool(faith["auc_cam"].mean() < faith["auc_rand"].mean() and win > 0.5)
print("  -> explanation is", "FAITHFUL (beats random deletion)" if faithful else "NOT clearly faithful: do not use these maps as evidence")

em = float(faith["edge_mass"].mean())
print(f"\nBorder audit: {em:.0%} of attention lies in the outer 10% border (uniform attention would give about 36%)")
if em > 0.50:
    print("  WARNING: the model attends mostly to the image border / background. This suggests a shortcut (scanner frame, text marks, collimation).")
else:
    print("  OK: attention is on the central hand region, not on the border.")

with open(RESULTS_DIR / f"{model_name}_xai_faithfulness.json", "w") as f:
    json.dump({"model_name": model_name, "split": XAI_SPLIT, "layer": XAI_LAYER, "target": XAI_TARGET, "n_images": int(len(faith)),
               "mean_auc_gradcampp": float(faith["auc_cam"].mean()), "mean_auc_random": float(faith["auc_rand"].mean()),
               "gradcampp_beats_random_frac": win, "wilcoxon_p": pval, "faithful": faithful, "mean_edge_attention": em},
              f, indent=2, default=to_python)
print("Saved:", RESULTS_DIR / f"{model_name}_xai_faithfulness.json")

### CHECK 5: Grad-CAM++

In [ ]:

from scipy.stats import spearmanr
import copy as _cp
_res5 = []
def _c5(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
    _res5.append(bool(passed))

_n5 = min(12, len(xai_y))
_idx5 = np.random.default_rng(SEED).choice(len(xai_y), size=_n5, replace=False)
cams5, probs5, cls5 = cams_for(_idx5)

_lay = get_target_layer(model, MODEL_NAME, XAI_LAYER == "fine")
_store = {}
_hk = _lay.register_forward_hook(lambda m, i, o: _store.__setitem__("a", o.detach()))
model.eval()
with torch.no_grad():
    model(xai_data[int(_idx5[0])][0][None].to(device))
_hk.remove()
_stride = 32 if XAI_LAYER == "last" else 16
_c5(f"hooked feature map is {imgSize // _stride}x{imgSize // _stride}", tuple(_store["a"].shape[-2:]) == (imgSize // _stride, imgSize // _stride), str(tuple(_store["a"].shape)))


_flat = cams5.reshape(_n5, -1)
_c5("heat-maps are finite and inside [0, 1]", bool(np.isfinite(cams5).all() and cams5.min() >= 0 and cams5.max() <= 1.0 + 1e-6), f"min {cams5.min():.2f} max {cams5.max():.2f}")
_c5("no empty heat-map (every map reaches 1.0 after normalisation)", bool((_flat.max(1) > 0.99).all()), f"{int((_flat.max(1) <= 0.99).sum())} empty")
_c5("heat-maps are not flat (std > 0.02)", bool((_flat.std(1) > 0.02).all()), f"min std {_flat.std(1).min():.3f}")

if XAI_TARGET == "class":
    _xb5 = torch.stack([xai_data[int(i)][0] for i in _idx5])
    _ca, _, _ = grad_campp(_xb5, cls=np.zeros(_n5, dtype=int))
    _cb, _, _ = grad_campp(_xb5, cls=np.full(_n5, len(class_names) - 1))
    _rho_cls = np.nanmean([spearmanr(a[::8, ::8].ravel(), b[::8, ::8].ravel())[0] for a, b in zip(_ca, _cb)])
    _c5("maps for the lowest and the highest grade differ (mean Spearman rho < 0.9)", _rho_cls < 0.9, f"rho = {_rho_cls:.2f}")


_rand = _cp.deepcopy(model)
torch.manual_seed(SEED + 1)
for _m in _rand.modules():
    if hasattr(_m, "reset_parameters"):
        _m.reset_parameters()
_orig_model = model
model = _rand.to(device).eval()
try:
    cams_rnd, _, _ = cams_for(_idx5)
finally:
    model = _orig_model
    del _rand
_rho_rnd = np.nanmean([spearmanr(a[::8, ::8].ravel(), b[::8, ::8].ravel())[0] for a, b in zip(cams5, cams_rnd)])
_c5("maps change when the weights are randomised (mean Spearman rho < 0.5)", _rho_rnd < 0.5, f"rho = {_rho_rnd:.2f}")

_edge = np.array([attention_summary(c)["edge_mass"] for c in cams5])
_c5("no sampled map has > 50% of its attention in the outer 10% border", bool((_edge < 0.5).all()), f"worst {_edge.max():.0%}")

_c5("deletion test: Grad-CAM++ beats random on > 50% of images and on average", bool(faithful), f"wins {win:.0%}, p = {pval}")
_c5("border audit (mean edge attention < 50%)", em < 0.5, f"{em:.0%}")

print("\nCHECK 5 RESULT:", "Grad-CAM++ is working." if all(_res5) else
      "Grad-CAM++ has a problem: do not use these maps as evidence until the FAIL lines are explained.")
print("Look at the overlays too: hot regions should sit on the carpal / wrist / finger joints, not on the black background, the 'R' marker, a ring or the image corners.")


In [ ]:

import importlib, subprocess, sys, gc as _gc

USE_VLM            = True
VLM_ID             = "Qwen/Qwen2-VL-2B-Instruct"
VLM_DTYPE          = "float16"
VLM_MAX_CASES      = 6
VLM_MAX_NEW_TOKENS = 200
FREE_VLM_AFTER     = True

def _ensure_transformers():
    try:
        import transformers
        from packaging.version import Version
        if Version(transformers.__version__) >= Version("4.45"):
            return True
    except Exception:
        pass
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.45", "accelerate"], check=False)
    importlib.invalidate_caches()
    try:
        import transformers
        return True
    except Exception as e:
        print("transformers could not be imported:", type(e).__name__)
        return False

vlm, vlm_processor = None, None
if USE_VLM and _ensure_transformers():
    try:
        from transformers import AutoProcessor
        try:
            from transformers import AutoModelForImageTextToText as _AutoVLM
        except ImportError:
            from transformers import AutoModelForVision2Seq as _AutoVLM
        _dt = {"float16": torch.float16, "float32": torch.float32, "bfloat16": torch.bfloat16}[VLM_DTYPE]
        if not torch.cuda.is_available():
            _dt = torch.float32
        try:
            vlm_processor = AutoProcessor.from_pretrained(VLM_ID, min_pixels=128 * 28 * 28, max_pixels=640 * 28 * 28) \
                if "qwen" in VLM_ID.lower() else AutoProcessor.from_pretrained(VLM_ID)
        except TypeError:
            vlm_processor = AutoProcessor.from_pretrained(VLM_ID)
        try:
            vlm = _AutoVLM.from_pretrained(VLM_ID, torch_dtype=_dt, low_cpu_mem_usage=True)
        except TypeError:
            vlm = _AutoVLM.from_pretrained(VLM_ID, dtype=_dt, low_cpu_mem_usage=True)
        vlm = vlm.to(device).eval()
        print(f"VLM loaded: {VLM_ID} ({_dt}) on {device}")
    except Exception as e:
        vlm, vlm_processor = None, None
        print(f"VLM could not be loaded ({type(e).__name__}: {str(e)[:200]}).")
        print("The next cell will use the template report instead. Check internet access / VLM_ID, then re-run this cell.")
else:
    print("VLM skipped (USE_VLM is False or transformers is unavailable): template reports will be used.")

### VLM helpers, check, reports


In [ ]:

DISCLAIMER = ("Decision-support only. The grade must be confirmed by a clinician; the heat-map shows where the model looked, "
              "not proven pathology.")
FINDINGS_LINE = ("Radiographic findings: not assessed here (the vision-language model is a general-purpose model that is not validated "
                 "for reading radiographs). A clinician must confirm the grade.")

def prob_str(rec):
    return ", ".join(f"{c} {rec['probs'][c]:.0%}" for c in class_names)

def facts_block(rec):

    conformal = ("ANSWERED: only one grade is plausible at the chosen confidence" if rec["answered"]
                 else "DEFERRED TO CLINICIAN: more than one grade (or none) is plausible")
    return (f"Model decision: {rec['pred']} ({rec['conf']:.0%}) | probabilities: {prob_str(rec)}\n"
            f"Conformal safety check: {conformal}\n"
            f"Attention (computed from the heat-map): {rec['summary']['sentence']}")

def build_prompt(rec):

    return (
        "You assist a rheumatology team. The image has two panels: on the left a hand/wrist X-ray, on the right the same X-ray with "
        "a colour overlay that shows where an AI grading model looked most (red/yellow = strongest). The colours only show where the "
        "AI looked; they say nothing about disease.\n\n"
        f"AI output (not a diagnosis): grade {rec['pred']} with {rec['conf']:.0%} confidence (probabilities: {prob_str(rec)}).\n"
        f"Safety check: {'answered automatically' if rec['answered'] else 'NOT certain enough, so the case is deferred to a clinician'}.\n"
        f"Heat-map facts: {rec['summary']['sentence']}.\n\n"
        "Write at most 80 words in exactly two short parts:\n"
        "A. Image: which anatomy is visible (hands, wrists, one or both sides) and whether the image quality looks adequate.\n"
        "B. Model: restate the AI grade, its confidence and where it looked, using only the facts above.\n"
        "Do not describe bone or joint abnormalities. Do not mention treatment or medication. Do not name any grade other than the AI grade."
    )

def template_text(rec):
    rel = ("The model was not certain enough, so a clinician should grade this image." if not rec["answered"]
           else "The model is reasonably certain, but a clinician should still confirm the grade.")
    return (f"A. Image: not described (template report; no vision-language model text was used).\n"
            f"B. Model: grade {rec['pred']} ({rec['conf']:.0%}); the heat-map peaks in the {rec['summary']['where']} of the image and is "
            f"{rec['summary']['spread']}. {rel}\n{FINDINGS_LINE}")

_FINDING_WORDS = ("erosion", "deformit", "subluxation", "ankylosis", "osteopenia", "osteoporosis", "narrowing", "swelling",
                  "bone density", "joint space", "joint-space", "bone destruction", "cartilage")
_TREAT_WORDS = ("nsaid", "medication", "treatment", "therapy", "methotrexate", "steroid", "surgery", "drug")
_DISEASE_FROM_HEATMAP = ("inflammation", "damage", "active disease")

def audit_vlm(text, rec, truncated):

    t = text.lower()
    hard, soft = [], []
    if truncated:
        hard.append("cut off at the token limit")
    if rec["pred"].lower() not in t:
        hard.append("does not state the model's grade")
    others = [c for c in class_names if c != rec["pred"] and
              re.search(rf"\b{c.lower()}\b\s+(rheumatoid|arthritis|disease|grade|ra\b|severity)", t)]
    if others:
        hard.append("names another grade: " + ", ".join(others))
    if any(w in t for w in _FINDING_WORDS):
        hard.append("claims radiographic findings (not verifiable)")
    if any(w in t for w in _TREAT_WORDS):
        hard.append("mentions treatment")
    if any(w in t for w in _DISEASE_FROM_HEATMAP):
        hard.append("reads the heat-map as disease")
    if len(text.split()) > 110:
        soft.append("longer than requested")
    return hard, soft

@torch.no_grad()
def vlm_generate(pil_img, prompt):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(text=[text], images=[pil_img], return_tensors="pt")
    inputs = {k: (v.to(vlm.device, dtype=vlm.dtype) if torch.is_floating_point(v) else v.to(vlm.device)) for k, v in inputs.items()}
    out = vlm.generate(**inputs, max_new_tokens=VLM_MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.1)
    gen = out[:, inputs["input_ids"].shape[1]:]
    truncated = gen.shape[1] >= VLM_MAX_NEW_TOKENS
    return vlm_processor.batch_decode(gen, skip_special_tokens=True)[0].strip(), truncated


In [ ]:

_res6 = []
def _c6(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
    _res6.append(bool(passed))

if vlm is None:
    print("CHECK 6a: the VLM is not loaded, so every report will be the code-built template (this is safe, just less rich).")
    print("EXPECTED when the VLM is loaded: 'VLM params: about 2.2B', dtype float16 on cuda. Re-run the VLM load cell and check the internet / VLM_ID.")
else:
    print(f"VLM params: {sum(p.numel() for p in vlm.parameters()) / 1e9:.2f}B | dtype {vlm.dtype} | device {vlm.device}")
    _a = np.full((384, 768, 3), 255, np.uint8); cv2.circle(_a, (192, 192), 120, (220, 0, 0), -1)
    _b = np.full((384, 768, 3), 255, np.uint8); cv2.rectangle(_b, (100, 90), (300, 290), (0, 0, 220), -1)
    _c = np.full((384, 768, 3), 255, np.uint8); cv2.rectangle(_c, (480, 90), (680, 290), (0, 160, 0), -1)
    tests6 = [(_a, "What colour is the large shape in this image? Answer with one word.", "red"),
              (_b, "Is the shape a circle, a square or a triangle? Answer with one word.", "square"),
              (_c, "Is the coloured shape on the left half or the right half of the image? Answer with one word.", "right")]
    for arr, q, want in tests6:
        ans, _ = vlm_generate(Image.fromarray(arr), q)
        _c6(f"VLM answers '{want}' for: {q[:55]}...", want in ans.lower(), f"got: {ans[:40]!r}")
    if not all(_res6):
        print("If these fail the model is not seeing the image (dtype / preprocessing problem): set VLM_DTYPE = 'float32' and re-run the load cell.")
    if "xai_records" in globals() and xai_records:
        _p = build_prompt(xai_records[0])
        _c6("prompt does not leak the true label", "true" not in _p.lower(), "")
        _c6("prompt lists no example findings (the small model copies lists)", not any(w in _p.lower() for w in _FINDING_WORDS), "")
    print("\nCHECK 6a RESULT:", "VLM loaded and sees images." if all(_res6) else "VLM problem: see FAIL lines (template reports will still work).")


In [ ]:

vlm_reports = []
for n_, rec in enumerate(xai_records[:VLM_MAX_CASES], 1):
    composite = Image.fromarray(np.concatenate([rec["img_u8"], rec["overlay"]], axis=1))
    text, source, raw_text, hard, soft, truncated = "", "template", "", [], [], False
    if vlm is not None:
        try:
            raw_text, truncated = vlm_generate(composite, build_prompt(rec))
            hard, soft = audit_vlm(raw_text, rec, truncated)
            if len(raw_text) >= 20 and not hard:
                text, source = raw_text + "\n" + FINDINGS_LINE, VLM_ID
            else:
                print(f"[case {n_}] VLM text rejected ({'; '.join(hard) or 'empty'}): template report used.")
        except Exception as e:
            print(f"[case {n_}] VLM generation failed ({type(e).__name__}: {str(e)[:150]}); using the template report.")
    if not text:
        text, source = template_text(rec), "template"

    comp_file = RESULTS_DIR / f"{model_name}_report_{n_}.png"
    composite.save(comp_file)
    vlm_reports.append({"case": n_, "image_index": rec["idx"], "image_path": rec["path"], "true_label_for_evaluation_only": rec["true"],
                        "model_prediction": rec["pred"], "probabilities": rec["probs"], "conformal_answered": rec["answered"],
                        "attention": rec["summary"]["sentence"], "explanation_source": source, "explanation": text,
                        "vlm_raw_text": raw_text, "vlm_hard_problems": hard, "vlm_soft_problems": soft, "vlm_truncated": truncated,
                        "composite_image": str(comp_file)})

    plt.figure(figsize=(8, 4.2))
    plt.imshow(composite)
    plt.axis("off")
    plt.title(f"Case {n_}: true {rec['true']} (evaluation only) | pred {rec['pred']} | "
              f"{'answered' if rec['answered'] else 'DEFERRED'}", fontsize=9)
    plt.show()
    print(facts_block(rec))
    print(f"\n--- explanation ({source}) ---\n{text}\n\n{DISCLAIMER}\n" + "=" * 90)

with open(RESULTS_DIR / f"{model_name}_xai_vlm_reports.json", "w") as f:
    json.dump({"model_name": model_name, "split": XAI_SPLIT, "xai_method": "Grad-CAM++", "xai_layer": XAI_LAYER,
               "xai_target": XAI_TARGET, "vlm": VLM_ID if vlm is not None else None, "reports": vlm_reports},
              f, indent=2, default=to_python)
print("Saved:", RESULTS_DIR / f"{model_name}_xai_vlm_reports.json")

if FREE_VLM_AFTER and vlm is not None:
    del vlm, vlm_processor
    vlm, vlm_processor = None, None
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("VLM unloaded from GPU memory.")


In [ ]:

_res6b = []
def _c6b(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "|", name, "|", detail)
    _res6b.append(bool(passed))

_n = len(vlm_reports)
_c6b("reports were generated", _n > 0, f"{_n} reports")
_c6b("every report names the model's grade", all(r["model_prediction"].lower() in r["explanation"].lower() for r in vlm_reports), "")
_c6b("every report carries the clinician-confirmation line", all("clinician must confirm" in r["explanation"].lower() for r in vlm_reports), "")
def _clean(r):
    t = r["explanation"].lower().replace(FINDINGS_LINE.lower(), "")
    return not any(w in t for w in _FINDING_WORDS + _TREAT_WORDS + _DISEASE_FROM_HEATMAP)
_c6b("no shown text claims findings, treatment, or disease read from the heat-map", all(_clean(r) for r in vlm_reports), "")
_used = [r for r in vlm_reports if r["explanation_source"] != "template"]
if any(r["vlm_raw_text"] for r in vlm_reports):
    _tried = sum(1 for r in vlm_reports if r["vlm_raw_text"])
    print(f"VLM text accepted by the audit: {len(_used)} of {_tried} generated "
          f"({len(_used) / max(_tried, 1):.0%}); truncated: {sum(r['vlm_truncated'] for r in vlm_reports)}")
    for r in vlm_reports:
        if r["vlm_hard_problems"]:
            print(f"  case {r['case']}: rejected -> {r['vlm_hard_problems']}")
print("\nCHECK 6b RESULT:", "Reports are consistent with the model output." if all(_res6b) else "Report problem: see FAIL lines.")
print("Reminder: the VLM never decides the grade; a general-purpose 2B model cannot be trusted to describe radiographic findings.")


## Test set evaluation

In [ ]:
if EVAL_TEST:
    y_test, probs_test = predict_loader(testLoader)
    metrics_test = compute_metrics(y_test, probs_test)
    ci_test = bootstrap_ci(y_test, probs_test)

    print("\n=== TEST SET RESULTS ===")
    print(classification_report(y_test, decode_probs(probs_test), target_names=class_names, digits=4))
    print(f"Macro-F1:         {metrics_test['macro_f1']:.4f}")
    print(f"QWK:              {metrics_test['qwk']:.4f}")
    print(f"Within one grade: {metrics_test['within_one_grade']:.4f}")
    print(f"MAE (grades):     {metrics_test['mae_grade']:.4f}")
    print_ci(ci_test, metrics_test)

    conf_test = deferral_report(y_test, probs_test, qhat_val)
    print(f"\n=== TEST: conformal deferral (alpha={alpha_star}, calibrated on val) ===")
    print(f"Answered: {conf_test['answered_frac']:.1%}  Deferred: {conf_test['deferred_frac']:.1%}  "
          f"Accuracy on answered: {conf_test['acc_answered']:.3f}  Set coverage: {conf_test['set_coverage']:.3f}")
    for c in class_names:
        pc = conf_test["per_class"][c]
        print(f"  {c:<9} recall on answered {pc['recall_answered']:.3f} | share of class answered {pc['frac_answered']:.1%}")
    test_file = RESULTS_DIR / f"{model_name}_test_results.json"
    with open(test_file, 'w') as f:
        json.dump({"model_name": model_name, "test": metrics_test, "test_bootstrap_ci": ci_test,
                   "conformal": {"alpha": alpha_star, "qhat": qhat_val.tolist(), **conf_test}},
                  f, indent=2, default=to_python)
    print(f"Saved: {test_file}")
    np.savez_compressed(RESULTS_DIR / f"{model_name}_test_probs.npz", y_true=y_test, probs=probs_test)
else:
    print("Set EVAL_TEST = True to evaluate the test set.")
